# SQL:DUNGEON Master Submission Notebook

**Exported:** 3/28/2026, 5:50:07 AM

This combined notebook is organized to reflect the project brief: individual SQLNOIR mysteries plus paired real-world problem-solving work.

- Kazi individual mysteries: 10
- Azm individual mysteries: 10
- Guild contract sessions: 10 (5 navigator, 5 driver)

Kazi and Azm are presented as the two individual student tracks, while Guild Contracts capture the shared navigator/driver sessions.

## Project Overview

The project gamifies T-SQL learning through detective-style mysteries and paired business problem solving.

- Strengthen T-SQL fluency through joins, subqueries, CTEs, functions, aggregations, and query optimization.
- Use SQLNOIR mystery framing so each query uncovers evidence instead of just returning rows.
- Document reasoning, code, and results in notebook form for Azure Data Studio, SSMS, or VS Code.
- Push beyond basics with advanced patterns such as window functions, recursive CTEs, or deeper query design when the mystery allows it.

## Individual Mystery Structure

- Each mystery uses four sections.
- Sections 1-3 gather evidence through incremental analysis.
- Section 4 should deliver the optimized final solution.
- Each notebook section should include narrative explanation, query work, and interpretation.

## Paired Session Structure

- Navigator defines and refines the business need.
- Driver writes and tests the T-SQL.
- The notebook should preserve query evolution, results, and final recommendations.

## Evaluation Reminders

- Correctness: the final query must solve the mystery accurately.
- Efficiency: the solution should avoid unnecessary work and show thoughtful query design.
- Documentation: the notebook should clearly explain the reasoning behind each section.
- Creativity: the notebook should show depth beyond a first-pass answer when possible.
- Collaboration: role alternation, listening, and requirement handoff stay clear throughout the session.
- Depth: the pair uses advanced T-SQL patterns such as CTEs, subqueries, ranking, pivots, or performance thinking.
- Innovation: the solution goes beyond a first-pass answer and explores edge cases, tuning, or richer analysis.
- Documentation: the notebook captures role notes, query evolution, evidence, and final recommendations.

## Kazi Individual SQLNOIR Mysteries

These 10 mysteries represent one full individual track of detective-style T-SQL work.

# Mystery #1

## The Merchant Who Never Ran Out

**Track:** Kazi Individual SQLNOIR Mysteries  
**Quest ID:** Q001  
**Rank:** Legendary  
**Focus Areas:** SELECT, JOIN, SUM, CTE, Window Functions  
**Exported:** 3/28/2026, 5:50:07 AM

*Narrative prompt*

The guild auditors have flagged a merchant in the Market Quarter whose stockrooms never seem to empty — no matter how much he sells, the shelves are always full. Someone is cooking the ledgers. Find the name of the merchant who defies the laws of supply.

**Track:** Kazi Individual SQLNOIR Mysteries  
**Quest ID:** Q001  
**Rank:** Legendary  
**Expected answer format:** The merchant's name, lowercase.

## Sample Database Schema

### `merchants`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |
| district | TEXT |  |

### `inventory`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| merchant_id | INTEGER |  |
| item | TEXT |  |
| stock | INTEGER |  |
| last_updated | TEXT |  |

### `orders`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| merchant_id | INTEGER |  |
| item | TEXT |  |
| quantity | INTEGER |  |
| order_date | TEXT |  |
| supplier | TEXT |  |

### `sales`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| merchant_id | INTEGER |  |
| item | TEXT |  |
| quantity | INTEGER |  |
| sale_date | TEXT |  |


## Section 1 - Part 1 — The Stockroom Anomaly

**Section Goal:** The auditors found something strange: certain stockrooms hold impossibly large inventories. Your task is to identify which merchants are sitting on suspiciously high stock levels for their district.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [1]:
SELECT m.name, i.item, i.stock
FROM merchants m
JOIN inventory i ON m.id = i.merchant_id
WHERE i.stock > 100
ORDER BY i.stock DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 2 - Part 2 — Supply vs. Demand

**Section Goal:** Now compare what each merchant ordered from suppliers versus what they actually sold. The gap will reveal who is operating beyond their supply chain.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [2]:
SELECT
  m.name,
  (SELECT COALESCE(SUM(o.quantity),0) FROM orders o WHERE o.merchant_id = m.id) AS total_ordered,
  (SELECT COALESCE(SUM(s.quantity),0) FROM sales s  WHERE s.merchant_id = m.id) AS total_sold
FROM merchants m
ORDER BY total_sold DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 3 - Part 3 — Phantom Stock Variance

**Section Goal:** Calculate the phantom stock: the difference between what was sold and what was ordered. A positive number means the merchant sold goods they never officially received.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [3]:
SELECT
  m.name,
  (SELECT COALESCE(SUM(s.quantity),0) FROM sales s WHERE s.merchant_id = m.id) -
  (SELECT COALESCE(SUM(o.quantity),0) FROM orders o WHERE o.merchant_id = m.id) AS phantom_stock
FROM merchants m
ORDER BY phantom_stock DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 4 - Part 4 — The Verdict

**Section Goal:** Compile all the evidence into one CTE. Name the merchant whose phantom stock variance is highest — the one who sold goods that never existed in the official ledger.

This section should combine the prior evidence into one final, optimized answer.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [4]:
WITH order_totals AS (
  SELECT merchant_id, SUM(quantity) AS total_ordered
  FROM orders GROUP BY merchant_id
),
sale_totals AS (
  SELECT merchant_id, SUM(quantity) AS total_sold
  FROM sales GROUP BY merchant_id
),
variance AS (
  SELECT
    m.name,
    COALESCE(st.total_sold, 0) - COALESCE(ot.total_ordered, 0) AS phantom_stock
  FROM merchants m
  LEFT JOIN order_totals ot ON m.id = ot.merchant_id
  LEFT JOIN sale_totals  st ON m.id = st.merchant_id
)
SELECT name, phantom_stock
FROM variance
WHERE phantom_stock > 0
ORDER BY phantom_stock DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain why this optimized final query solves the mystery completely.

## Final Reflection

**Final answer format:** The merchant's name, lowercase.

- State the final answer clearly.
- Explain why the evidence across all four sections supports that answer.
- Briefly note how correctness, efficiency, documentation, and creativity show up in the finished work.

---
*Generated by SQL:DUNGEON - CSCI 331 - Queens College CUNY*

# Mystery #2

## The Curse of the Phantom Order

**Track:** Kazi Individual SQLNOIR Mysteries  
**Quest ID:** Q002  
**Rank:** Rare  
**Focus Areas:** JOIN, LEFT JOIN, Recursive CTE, NULL checks  
**Exported:** 3/28/2026, 5:50:07 AM

*Narrative prompt*

Shipments are arriving at addresses that have no corresponding paid orders. Someone is receiving guild cargo without paying — but the trail is buried in a chain of sub-orders that loops back on itself. Trace the curse to its origin.

**Track:** Kazi Individual SQLNOIR Mysteries  
**Quest ID:** Q002  
**Rank:** Rare  
**Expected answer format:** The customer's name, lowercase with underscore.

## Sample Database Schema

### `customers`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |
| guild | TEXT |  |
| registered | INTEGER |  |

### `orders`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| customer_id | INTEGER |  |
| parent_order_id | INTEGER |  |
| item | TEXT |  |
| amount | INTEGER |  |
| paid | INTEGER |  |
| order_date | TEXT |  |

### `shipments`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| order_id | INTEGER |  |
| delivered_to | TEXT |  |
| delivered_date | TEXT |  |
| quantity | INTEGER |  |


## Section 1 - Part 1 — The Missing Bill

**Section Goal:** The shipping manifest shows deliveries that were never invoiced. Find all shipments whose linked order was never paid.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [1]:
SELECT s.id AS shipment_id, s.delivered_to, s.quantity, s.delivered_date
FROM shipments s
JOIN orders o ON s.order_id = o.id
WHERE o.paid = 0
ORDER BY s.delivered_date;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 2 - Part 2 — The Recipient

**Section Goal:** Now trace those unpaid shipments back to the customer who placed the orders. Who registered to receive these ghost deliveries?

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [2]:
SELECT c.name, s.delivered_to, s.quantity, s.delivered_date
FROM shipments s
JOIN orders o   ON s.order_id = o.id
JOIN customers c ON o.customer_id = c.id
WHERE o.paid = 0
ORDER BY s.delivered_date;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 3 - Part 3 — The Order Chain

**Section Goal:** The orders are nested — sub-orders reference parent orders that themselves reference parents. Use a recursive CTE to unravel the full chain.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [3]:
WITH RECURSIVE order_chain AS (
  SELECT id, customer_id, parent_order_id, item, paid, 0 AS depth
  FROM orders WHERE parent_order_id IS NULL
  UNION ALL
  SELECT o.id, o.customer_id, o.parent_order_id, o.item, o.paid, oc.depth + 1
  FROM orders o
  JOIN order_chain oc ON o.parent_order_id = oc.id
)
SELECT c.name, oc.id AS order_id, oc.depth, oc.paid
FROM order_chain oc
JOIN customers c ON oc.customer_id = c.id
ORDER BY oc.depth, oc.id;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 4 - Part 4 — The Verdict

**Section Goal:** Combine the recursive order chain with shipment data to conclusively name the client who received guild cargo without a single paid invoice.

This section should combine the prior evidence into one final, optimized answer.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [4]:
WITH RECURSIVE order_chain AS (
  SELECT id, customer_id, parent_order_id, paid, 0 AS depth
  FROM orders WHERE parent_order_id IS NULL
  UNION ALL
  SELECT o.id, o.customer_id, o.parent_order_id, o.paid, oc.depth + 1
  FROM orders o JOIN order_chain oc ON o.parent_order_id = oc.id
),
ghost_orders AS (
  SELECT oc.id AS order_id, oc.customer_id
  FROM order_chain oc
  WHERE oc.paid = 0
)
SELECT DISTINCT c.name AS suspect
FROM ghost_orders go
JOIN customers c  ON go.customer_id = c.id
JOIN shipments s  ON s.order_id = go.order_id
ORDER BY suspect;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain why this optimized final query solves the mystery completely.

## Final Reflection

**Final answer format:** The customer's name, lowercase with underscore.

- State the final answer clearly.
- Explain why the evidence across all four sections supports that answer.
- Briefly note how correctness, efficiency, documentation, and creativity show up in the finished work.

---
*Generated by SQL:DUNGEON - CSCI 331 - Queens College CUNY*

# Mystery #3

## The Shapeshifter's Ledger

**Track:** Kazi Individual SQLNOIR Mysteries  
**Quest ID:** Q003  
**Rank:** Uncommon  
**Focus Areas:** GROUP BY, HAVING, JOIN, CASE WHEN, CTE, PIVOT  
**Exported:** 3/28/2026, 5:50:07 AM

*Narrative prompt*

The guild's payroll office has discovered they're paying for more workers than they hired. Someone has registered multiple identities — each drawing a full wage — but only one person is logging the hours. Find the shapeshifter who fractured their identity across the ledger.

**Track:** Kazi Individual SQLNOIR Mysteries  
**Quest ID:** Q003  
**Rank:** Uncommon  
**Expected answer format:** The employee's base name, lowercase.

## Sample Database Schema

### `employees`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| alias | TEXT |  |
| base_name | TEXT |  |
| department | TEXT |  |
| hire_date | TEXT |  |

### `payroll`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| employee_id | INTEGER |  |
| month | TEXT |  |
| wage | INTEGER |  |

### `work_logs`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| employee_id | INTEGER |  |
| work_date | TEXT |  |
| hours | INTEGER |  |
| task | TEXT |  |


## Section 1 - Part 1 — The Duplicate Register

**Section Goal:** The employee register contains aliases — short names with numeric suffixes. Group by the base name to find which identity appears more than once on the payroll.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [1]:
SELECT base_name, COUNT(*) AS alias_count, GROUP_CONCAT(alias) AS aliases
FROM employees
GROUP BY base_name
HAVING COUNT(*) > 1
ORDER BY alias_count DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 2 - Part 2 — Triple Wages

**Section Goal:** Each alias receives a full wage independently. Join payroll to employees and see what each identity is collecting — then look at the base_name totals.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [2]:
SELECT e.base_name, e.alias, SUM(p.wage) AS total_paid
FROM employees e
JOIN payroll p ON e.id = p.employee_id
GROUP BY e.id, e.base_name, e.alias
ORDER BY e.base_name, total_paid DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 3 - Part 3 — The Absent Witnesses

**Section Goal:** Someone collecting triple wages should be logging triple hours. Pivot the wage payments by month using CASE WHEN. Then check work_logs — most of these aliases have no hours at all.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [3]:
SELECT
  e.base_name,
  SUM(CASE WHEN p.month = '1247-01' THEN p.wage ELSE 0 END) AS jan,
  SUM(CASE WHEN p.month = '1247-02' THEN p.wage ELSE 0 END) AS feb,
  SUM(CASE WHEN p.month = '1247-03' THEN p.wage ELSE 0 END) AS mar,
  SUM(p.wage) AS total
FROM employees e
JOIN payroll p ON e.id = p.employee_id
GROUP BY e.base_name
ORDER BY total DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 4 - Part 4 — The Verdict

**Section Goal:** Combine the identity map with payroll totals in a CTE. The shapeshifter is the base_name with multiple aliases, outsized wages, and only one alias that ever showed up to work.

This section should combine the prior evidence into one final, optimized answer.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [4]:
WITH identity_map AS (
  SELECT base_name,
    COUNT(*)            AS aliases,
    GROUP_CONCAT(alias) AS all_aliases
  FROM employees
  GROUP BY base_name
),
wage_totals AS (
  SELECT e.base_name, SUM(p.wage) AS total_wage
  FROM employees e
  JOIN payroll p ON e.id = p.employee_id
  GROUP BY e.base_name
),
log_totals AS (
  SELECT e.base_name, COALESCE(SUM(wl.hours), 0) AS total_hours
  FROM employees e
  LEFT JOIN work_logs wl ON e.id = wl.employee_id
  GROUP BY e.base_name
)
SELECT im.base_name, im.aliases, im.all_aliases, wt.total_wage, lt.total_hours
FROM identity_map im
JOIN wage_totals  wt ON im.base_name = wt.base_name
JOIN log_totals   lt ON im.base_name = lt.base_name
WHERE im.aliases > 1
ORDER BY wt.total_wage DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain why this optimized final query solves the mystery completely.

## Final Reflection

**Final answer format:** The employee's base name, lowercase.

- State the final answer clearly.
- Explain why the evidence across all four sections supports that answer.
- Briefly note how correctness, efficiency, documentation, and creativity show up in the finished work.

---
*Generated by SQL:DUNGEON - CSCI 331 - Queens College CUNY*

# Mystery #4

## The Clockwork Alibi

**Track:** Kazi Individual SQLNOIR Mysteries  
**Quest ID:** Q004  
**Rank:** Rare  
**Focus Areas:** ROW_NUMBER, LAG, RANK, Window Functions, CTE  
**Exported:** 3/28/2026, 5:50:07 AM

*Narrative prompt*

A theft struck the vault at '1247-02-14 02:30'. Every guard filed patrol logs. But window functions tell a different story — someone was absent exactly when it mattered. Sequence the patrols, measure the gaps, and find the guard who vanished.

**Track:** Kazi Individual SQLNOIR Mysteries  
**Quest ID:** Q004  
**Rank:** Rare  
**Expected answer format:** The guard's name, lowercase.

## Sample Database Schema

### `guards`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |
| post | TEXT |  |

### `patrol_log`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| guard_id | INTEGER |  |
| post | TEXT |  |
| logged_at | TEXT |  |

### `incidents`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| post | TEXT |  |
| occurred_at | TEXT |  |
| severity | TEXT |  |


## Section 1 - Part 1 — The Patrol Sequence

**Section Goal:** Every guard's log entries must be numbered in order to detect gaps. Use ROW_NUMBER() to assign a sequence to each guard's patrol entries.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [1]:
SELECT
  g.name,
  pl.post,
  pl.logged_at,
  ROW_NUMBER() OVER (PARTITION BY pl.guard_id ORDER BY pl.logged_at) AS seq
FROM patrol_log pl
JOIN guards g ON g.id = pl.guard_id
ORDER BY g.name, pl.logged_at;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 2 - Part 2 — Measuring the Gap

**Section Goal:** Use LAG() to fetch the previous check-in time and compute the gap in hours using julianday arithmetic.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [2]:
SELECT
  g.name,
  pl.post,
  pl.logged_at,
  LAG(pl.logged_at) OVER (PARTITION BY pl.guard_id ORDER BY pl.logged_at) AS prev_logged,
  ROUND(
    (julianday(pl.logged_at) - julianday(
      LAG(pl.logged_at) OVER (PARTITION BY pl.guard_id ORDER BY pl.logged_at)
    )) * 24, 2
  ) AS gap_hours
FROM patrol_log pl
JOIN guards g ON g.id = pl.guard_id
ORDER BY gap_hours DESC NULLS LAST;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 3 - Part 3 — Ranking the Suspects

**Section Goal:** Aggregate the worst gap per guard and use RANK() to surface who had the most suspicious absence.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [3]:
WITH gaps AS (
  SELECT
    g.name,
    pl.post,
    pl.logged_at,
    LAG(pl.logged_at) OVER (PARTITION BY pl.guard_id ORDER BY pl.logged_at) AS prev_logged,
    (julianday(pl.logged_at) - julianday(
      LAG(pl.logged_at) OVER (PARTITION BY pl.guard_id ORDER BY pl.logged_at)
    )) * 24 AS gap_hours
  FROM patrol_log pl
  JOIN guards g ON g.id = pl.guard_id
)
SELECT
  name,
  ROUND(MAX(gap_hours), 2) AS worst_gap,
  RANK() OVER (ORDER BY MAX(gap_hours) DESC) AS suspicion_rank
FROM gaps
WHERE gap_hours IS NOT NULL
GROUP BY name
ORDER BY suspicion_rank;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 4 - Part 4 — The Verdict

**Section Goal:** Confirm the culprit: find the guard whose patrol gap spans the exact time of the critical incident at vault_corridor.

This section should combine the prior evidence into one final, optimized answer.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [4]:
WITH patrol_gaps AS (
  SELECT
    g.id AS guard_id,
    g.name,
    pl.post,
    LAG(pl.logged_at) OVER (PARTITION BY pl.guard_id ORDER BY pl.logged_at) AS prev_logged,
    pl.logged_at,
    (julianday(pl.logged_at) - julianday(
      LAG(pl.logged_at) OVER (PARTITION BY pl.guard_id ORDER BY pl.logged_at)
    )) * 24 AS gap_hours
  FROM patrol_log pl
  JOIN guards g ON g.id = pl.guard_id
),
incident_cover AS (
  SELECT pg.name, pg.post, pg.prev_logged, pg.logged_at, pg.gap_hours, i.occurred_at, i.severity
  FROM patrol_gaps pg
  JOIN incidents i ON i.post = pg.post
  WHERE pg.prev_logged < i.occurred_at
    AND pg.logged_at  > i.occurred_at
    AND i.severity = 'critical'
)
SELECT name, post, prev_logged, logged_at, gap_hours, occurred_at
FROM incident_cover
ORDER BY gap_hours DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain why this optimized final query solves the mystery completely.

## Final Reflection

**Final answer format:** The guard's name, lowercase.

- State the final answer clearly.
- Explain why the evidence across all four sections supports that answer.
- Briefly note how correctness, efficiency, documentation, and creativity show up in the finished work.

---
*Generated by SQL:DUNGEON - CSCI 331 - Queens College CUNY*

# Mystery #5

## The Alchemist's Debt

**Track:** Kazi Individual SQLNOIR Mysteries  
**Quest ID:** Q005  
**Rank:** Common  
**Focus Areas:** julianday, Date Arithmetic, CASE WHEN, CTE, GROUP BY  
**Exported:** 3/28/2026, 5:50:07 AM

*Narrative prompt*

The Collector's Guild keeps meticulous debt records. Someone has been dodging payment for so long the debt has compounded beyond reason. Use date arithmetic to surface the oldest unpaid obligation and name the debtor who owes the most.

**Track:** Kazi Individual SQLNOIR Mysteries  
**Quest ID:** Q005  
**Rank:** Common  
**Expected answer format:** The debtor's name, lowercase.

## Sample Database Schema

### `debtors`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |
| guild | TEXT |  |
| district | TEXT |  |

### `loans`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| debtor_id | INTEGER |  |
| principal | INTEGER |  |
| issued_date | TEXT |  |
| due_date | TEXT |  |

### `payments`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| loan_id | INTEGER |  |
| amount_paid | INTEGER |  |
| payment_date | TEXT |  |


## Section 1 - Part 1 — Past Due

**Section Goal:** The registry lists every loan's due date. Filter down to the loans already past due as of '1247-03-15'.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [1]:
SELECT d.name, l.id AS loan_id, l.principal, l.due_date
FROM loans l
JOIN debtors d ON d.id = l.debtor_id
WHERE l.due_date < '1247-03-15'
ORDER BY l.due_date ASC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 2 - Part 2 — Days Overdue

**Section Goal:** Calculate exactly how many days overdue each loan is using julianday arithmetic.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [2]:
SELECT
  d.name,
  l.id AS loan_id,
  l.due_date,
  CAST(julianday('1247-03-15') - julianday(l.due_date) AS INTEGER) AS days_overdue
FROM loans l
JOIN debtors d ON d.id = l.debtor_id
WHERE l.due_date < '1247-03-15'
ORDER BY days_overdue DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 3 - Part 3 — Aging Buckets

**Section Goal:** Classify each overdue loan into an aging bucket: 0-30 days, 31-60 days, 61-90 days, or 90+ days.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [3]:
SELECT
  d.name,
  l.id AS loan_id,
  CAST(julianday('1247-03-15') - julianday(l.due_date) AS INTEGER) AS days_overdue,
  CASE
    WHEN julianday('1247-03-15') - julianday(l.due_date) <= 30  THEN '0-30'
    WHEN julianday('1247-03-15') - julianday(l.due_date) <= 60  THEN '31-60'
    WHEN julianday('1247-03-15') - julianday(l.due_date) <= 90  THEN '61-90'
    ELSE '90+'
  END AS aging_bucket
FROM loans l
JOIN debtors d ON d.id = l.debtor_id
WHERE l.due_date < '1247-03-15'
ORDER BY days_overdue DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 4 - Part 4 — The Verdict

**Section Goal:** Build a CTE that calculates outstanding balance per debtor (principal minus total payments) and rank by who owes the most.

This section should combine the prior evidence into one final, optimized answer.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [4]:
WITH paid AS (
  SELECT loan_id, COALESCE(SUM(amount_paid), 0) AS total_paid
  FROM payments
  GROUP BY loan_id),
balances AS (
  SELECT
    d.name,
    l.id AS loan_id,
    l.principal,
    COALESCE(p.total_paid, 0) AS total_paid,
    l.principal - COALESCE(p.total_paid, 0) AS outstanding
  FROM loans l
  JOIN debtors d ON d.id = l.debtor_id
  LEFT JOIN paid p ON p.loan_id = l.id
)
SELECT name, SUM(outstanding) AS total_outstanding
FROM balances
GROUP BY name
HAVING SUM(outstanding) > 0
ORDER BY total_outstanding DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain why this optimized final query solves the mystery completely.

## Final Reflection

**Final answer format:** The debtor's name, lowercase.

- State the final answer clearly.
- Explain why the evidence across all four sections supports that answer.
- Briefly note how correctness, efficiency, documentation, and creativity show up in the finished work.

---
*Generated by SQL:DUNGEON - CSCI 331 - Queens College CUNY*

# Mystery #6

## The Bloodline Curse

**Track:** Kazi Individual SQLNOIR Mysteries  
**Quest ID:** Q006  
**Rank:** Legendary  
**Focus Areas:** Recursive CTE, Self-Join, GROUP BY, CTE, Hierarchy  
**Exported:** 3/28/2026, 5:50:07 AM

*Narrative prompt*

The Verenthis bloodline carries a curse — every generation incurs more debt than it inherits assets. Use a recursive CTE to traverse the family tree, accumulate lineage debt, and name the dynasty drowning in obligation.

**Track:** Kazi Individual SQLNOIR Mysteries  
**Quest ID:** Q006  
**Rank:** Legendary  
**Expected answer format:** The dynasty root name, lowercase.

## Sample Database Schema

### `nobles`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |
| parent_id | INTEGER |  |
| estate_value | INTEGER |  |
| debt | INTEGER |  |


## Section 1 - Part 1 — The Family Tree

**Section Goal:** Begin with a simple self-join to display each noble alongside their parent, revealing one generation of the bloodline.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [1]:
SELECT
  child.name  AS noble,
  parent.name AS parent,
  child.debt,
  child.estate_value
FROM nobles child
LEFT JOIN nobles parent ON parent.id = child.parent_id
ORDER BY parent.name NULLS FIRST, child.name;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 2 - Part 2 — The Recursive Walk

**Section Goal:** Use a recursive CTE to walk the entire tree from root to leaf, tracking generation depth and the lineage path.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [2]:
WITH RECURSIVE lineage AS (
  SELECT id, name, parent_id, 0 AS generation,
         name AS path
  FROM nobles WHERE parent_id IS NULL
  UNION ALL
  SELECT n.id, n.name, n.parent_id, l.generation + 1,
         l.path || ' > ' || n.name
  FROM nobles n
  JOIN lineage l ON n.parent_id = l.id
)
SELECT name, generation, path
FROM lineage
ORDER BY path;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 3 - Part 3 — Subtree Debt

**Section Goal:** For each noble, calculate the total debt of their entire subtree — all descendants plus themselves.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [3]:
WITH RECURSIVE subtree(root_id, member_id, debt, estate_value) AS (
  SELECT id, id, debt, estate_value FROM nobles
  UNION ALL
  SELECT st.root_id, n.id, n.debt, n.estate_value
  FROM nobles n
  JOIN subtree st ON n.parent_id = st.member_id
)
SELECT
  r.name AS dynasty_root,
  SUM(st.debt)         AS total_debt,
  SUM(st.estate_value) AS total_estates
FROM subtree st
JOIN nobles r ON r.id = st.root_id
WHERE r.parent_id IS NULL
GROUP BY st.root_id, r.name
ORDER BY total_debt DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 4 - Part 4 — The Verdict

**Section Goal:** Summarize each dynasty: member count, total debt, total estates, and debt-to-estate ratio. Name the cursed bloodline.

This section should combine the prior evidence into one final, optimized answer.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [4]:
WITH RECURSIVE subtree(root_id, member_id, debt, estate_value) AS (
  SELECT id, id, debt, estate_value FROM nobles
  UNION ALL
  SELECT st.root_id, n.id, n.debt, n.estate_value
  FROM nobles n
  JOIN subtree st ON n.parent_id = st.member_id
)
SELECT
  r.name AS dynasty,
  COUNT(DISTINCT st.member_id)                               AS members,
  SUM(st.debt)                                              AS total_debt,
  SUM(st.estate_value)                                      AS total_estates,
  ROUND(CAST(SUM(st.debt) AS REAL) / SUM(st.estate_value) * 100, 1) AS debt_ratio_pct
FROM subtree st
JOIN nobles r ON r.id = st.root_id
WHERE r.parent_id IS NULL
GROUP BY st.root_id, r.name
ORDER BY debt_ratio_pct DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain why this optimized final query solves the mystery completely.

## Final Reflection

**Final answer format:** The dynasty root name, lowercase.

- State the final answer clearly.
- Explain why the evidence across all four sections supports that answer.
- Briefly note how correctness, efficiency, documentation, and creativity show up in the finished work.

---
*Generated by SQL:DUNGEON - CSCI 331 - Queens College CUNY*

# Mystery #7

## The Hollow Witness

**Track:** Kazi Individual SQLNOIR Mysteries  
**Quest ID:** Q007  
**Rank:** Uncommon  
**Focus Areas:** EXISTS, NOT EXISTS, Correlated Subquery, GROUP_CONCAT, CTE  
**Exported:** 3/28/2026, 5:50:07 AM

*Narrative prompt*

Four crimes. Multiple registered witnesses. Yet one 'witness' listed at the scene of every crime has never produced a single statement. Find the phantom who was everywhere and said nothing.

**Track:** Kazi Individual SQLNOIR Mysteries  
**Quest ID:** Q007  
**Rank:** Uncommon  
**Expected answer format:** The witness name, lowercase with underscore.

## Sample Database Schema

### `crimes`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| title | TEXT |  |
| location | TEXT |  |
| crime_date | TEXT |  |

### `witnesses`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |
| district | TEXT |  |
| registered | INTEGER |  |

### `testimonies`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| witness_id | INTEGER |  |
| crime_id | INTEGER |  |
| statement | TEXT |  |
| recorded_date | TEXT |  |

### `crime_witnesses`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| crime_id | INTEGER |  |
| witness_id | INTEGER |  |


## Section 1 - Part 1 — The Witness Registry

**Section Goal:** Find all witnesses who appear in the crime_witnesses table — those officially linked to at least one crime.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [1]:
SELECT DISTINCT w.name, w.district
FROM witnesses w
WHERE w.id IN (
  SELECT witness_id FROM crime_witnesses
)
ORDER BY w.name;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 2 - Part 2 — The Testified

**Section Goal:** Now filter to witnesses who have produced at least one testimony using EXISTS.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [2]:
SELECT w.name, w.district
FROM witnesses w
WHERE EXISTS (
  SELECT 1 FROM testimonies t
  WHERE t.witness_id = w.id
)
ORDER BY w.name;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 3 - Part 3 — The Silent Ones

**Section Goal:** Invert the EXISTS: find witnesses linked to crimes but with NO testimony on record.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [3]:
SELECT w.name, w.district
FROM witnesses w
WHERE w.id IN (SELECT witness_id FROM crime_witnesses)
  AND NOT EXISTS (
    SELECT 1 FROM testimonies t
    WHERE t.witness_id = w.id
  )
ORDER BY w.name;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 4 - Part 4 — The Verdict

**Section Goal:** Confirm the phantom: build two CTEs — ghost_witnesses (no testimony) and their crime list — to produce a full report.

This section should combine the prior evidence into one final, optimized answer.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [4]:
WITH ghost_witnesses AS (
  SELECT w.id, w.name, w.district
  FROM witnesses w
  WHERE w.id IN (SELECT witness_id FROM crime_witnesses)
    AND NOT EXISTS (
      SELECT 1 FROM testimonies t WHERE t.witness_id = w.id
    )
),
ghost_crimes AS (
  SELECT
    gw.name,
    COUNT(cw.crime_id)          AS crimes_linked,
    GROUP_CONCAT(c.title, ', ') AS crime_list
  FROM ghost_witnesses gw
  JOIN crime_witnesses cw ON cw.witness_id = gw.id
  JOIN crimes c           ON c.id = cw.crime_id
  GROUP BY gw.id, gw.name
)
SELECT name, crimes_linked, crime_list
FROM ghost_crimes
ORDER BY crimes_linked DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain why this optimized final query solves the mystery completely.

## Final Reflection

**Final answer format:** The witness name, lowercase with underscore.

- State the final answer clearly.
- Explain why the evidence across all four sections supports that answer.
- Briefly note how correctness, efficiency, documentation, and creativity show up in the finished work.

---
*Generated by SQL:DUNGEON - CSCI 331 - Queens College CUNY*

# Mystery #8

## The Counterfeit Ledger

**Track:** Kazi Individual SQLNOIR Mysteries  
**Quest ID:** Q008  
**Rank:** Rare  
**Focus Areas:** CASE WHEN, PIVOT, GROUP BY, CTE, Discrepancy Analysis  
**Exported:** 3/28/2026, 5:50:07 AM

*Narrative prompt*

The Royal Mint Inspectorate has found a discrepancy between coins struck and coins reported. Someone is skimming from production. Pivot the monthly ledger to expose the pattern — and name the mint hiding the shortfall.

**Track:** Kazi Individual SQLNOIR Mysteries  
**Quest ID:** Q008  
**Rank:** Rare  
**Expected answer format:** The mint name, lowercase.

## Sample Database Schema

### `mints`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |
| district | TEXT |  |
| licensed | INTEGER |  |

### `production`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| mint_id | INTEGER |  |
| month | TEXT |  |
| coins_struck | INTEGER |  |
| coins_reported | INTEGER |  |

### `inspections`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| mint_id | INTEGER |  |
| inspection_date | TEXT |  |
| coins_counted | INTEGER |  |
| passed | INTEGER |  |


## Section 1 - Part 1 — The Totals

**Section Goal:** Start with a basic GROUP BY to compare total coins struck versus total coins reported per mint.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [1]:
SELECT
  m.name,
  SUM(p.coins_struck)   AS total_struck,
  SUM(p.coins_reported) AS total_reported,
  SUM(p.coins_struck) - SUM(p.coins_reported) AS total_gap
FROM production p
JOIN mints m ON m.id = p.mint_id
GROUP BY p.mint_id, m.name
ORDER BY total_gap DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 2 - Part 2 — The Monthly Pivot

**Section Goal:** Pivot the production table to show struck and reported columns for each month side by side.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [2]:
SELECT
  m.name,
  SUM(CASE WHEN p.month = '1247-01' THEN p.coins_struck   ELSE 0 END) AS jan_struck,
  SUM(CASE WHEN p.month = '1247-01' THEN p.coins_reported ELSE 0 END) AS jan_reported,
  SUM(CASE WHEN p.month = '1247-02' THEN p.coins_struck   ELSE 0 END) AS feb_struck,
  SUM(CASE WHEN p.month = '1247-02' THEN p.coins_reported ELSE 0 END) AS feb_reported,
  SUM(CASE WHEN p.month = '1247-03' THEN p.coins_struck   ELSE 0 END) AS mar_struck,
  SUM(CASE WHEN p.month = '1247-03' THEN p.coins_reported ELSE 0 END) AS mar_reported
FROM production p
JOIN mints m ON m.id = p.mint_id
GROUP BY p.mint_id, m.name
ORDER BY m.name;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 3 - Part 3 — The Discrepancy Pivot

**Section Goal:** Now pivot the discrepancy itself — (coins_struck - coins_reported) per month — to show the skimming pattern month by month.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [3]:
SELECT
  m.name,
  SUM(CASE WHEN p.month = '1247-01' THEN p.coins_struck - p.coins_reported ELSE 0 END) AS jan_gap,
  SUM(CASE WHEN p.month = '1247-02' THEN p.coins_struck - p.coins_reported ELSE 0 END) AS feb_gap,
  SUM(CASE WHEN p.month = '1247-03' THEN p.coins_struck - p.coins_reported ELSE 0 END) AS mar_gap,
  SUM(p.coins_struck - p.coins_reported) AS total_gap
FROM production p
JOIN mints m ON m.id = p.mint_id
GROUP BY p.mint_id, m.name
ORDER BY total_gap DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 4 - Part 4 — The Verdict

**Section Goal:** Compare total reported against physical inspection counts to confirm the skimming. Name the mint whose reported total doesn't match what the inspectors found.

This section should combine the prior evidence into one final, optimized answer.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [4]:
WITH monthly_pivot AS (
  SELECT
    mint_id,
    SUM(coins_reported) AS total_reported
  FROM production
  GROUP BY mint_id
),
inspection_check AS (
  SELECT
    m.name,
    mp.total_reported,
    i.coins_counted,
    i.coins_counted - mp.total_reported AS inspection_gap,
    i.passed
  FROM monthly_pivot mp
  JOIN mints m        ON m.id = mp.mint_id
  JOIN inspections i  ON i.mint_id = mp.mint_id
)
SELECT name, total_reported, coins_counted, inspection_gap
FROM inspection_check
WHERE inspection_gap > 0
ORDER BY inspection_gap DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain why this optimized final query solves the mystery completely.

## Final Reflection

**Final answer format:** The mint name, lowercase.

- State the final answer clearly.
- Explain why the evidence across all four sections supports that answer.
- Briefly note how correctness, efficiency, documentation, and creativity show up in the finished work.

---
*Generated by SQL:DUNGEON - CSCI 331 - Queens College CUNY*

# Mystery #9

## The Smuggler's Cipher

**Track:** Kazi Individual SQLNOIR Mysteries  
**Quest ID:** Q009  
**Rank:** Uncommon  
**Focus Areas:** Self-Join, Circular Route, CTE, UNION, GROUP BY  
**Exported:** 3/28/2026, 5:50:07 AM

*Narrative prompt*

Contraband is moving through an unregistered port that loops between the same two docks to avoid guild checkpoints. A self-join reveals the hub of this circular network. Find the port at the center of the web.

**Track:** Kazi Individual SQLNOIR Mysteries  
**Quest ID:** Q009  
**Rank:** Uncommon  
**Expected answer format:** The port name, lowercase with underscore.

## Sample Database Schema

### `ports`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |
| region | TEXT |  |
| guild_controlled | INTEGER |  |

### `routes`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| from_port_id | INTEGER |  |
| to_port_id | INTEGER |  |
| distance_leagues | INTEGER |  |
| tariff_rate | REAL |  |
| registered | INTEGER |  |


## Section 1 - Part 1 — The Route Map

**Section Goal:** Self-join the routes table to ports to display full route names — from port to port — with registration status.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [1]:
SELECT
  r.id AS route_id,
  p1.name AS from_port,
  p2.name AS to_port,
  r.distance_leagues,
  r.registered
FROM routes r
JOIN ports p1 ON p1.id = r.from_port_id
JOIN ports p2 ON p2.id = r.to_port_id
ORDER BY r.registered, r.id;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 2 - Part 2 — Suspicious Ports

**Section Goal:** Filter to unregistered routes and flag any non-guild-controlled ports as 'suspicious'.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [2]:
SELECT
  r.id AS route_id,
  p1.name AS from_port,
  p2.name AS to_port,
  CASE WHEN p1.guild_controlled = 0 THEN 'suspicious' ELSE 'clear' END AS from_status,
  CASE WHEN p2.guild_controlled = 0 THEN 'suspicious' ELSE 'clear' END AS to_status
FROM routes r
JOIN ports p1 ON p1.id = r.from_port_id
JOIN ports p2 ON p2.id = r.to_port_id
WHERE r.registered = 0
ORDER BY r.id;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 3 - Part 3 — The Circular Loop

**Section Goal:** Find route pairs where A→B and B→A both exist as unregistered routes — the hallmark of a smuggling circuit.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [3]:
SELECT
  p1.name AS port_a,
  p2.name AS port_b,
  r1.distance_leagues
FROM routes r1
JOIN routes r2
  ON r2.from_port_id = r1.to_port_id
 AND r2.to_port_id   = r1.from_port_id
JOIN ports p1 ON p1.id = r1.from_port_id
JOIN ports p2 ON p2.id = r1.to_port_id
WHERE r1.registered = 0
  AND r2.registered = 0
  AND r1.from_port_id < r1.to_port_id
ORDER BY port_a;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 4 - Part 4 — The Verdict

**Section Goal:** Count how many unregistered routes each port appears in (as either origin or destination). The port with the most is the smuggling hub.

This section should combine the prior evidence into one final, optimized answer.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [4]:
WITH unregistered AS (
  SELECT from_port_id AS port_id FROM routes WHERE registered = 0
  UNION ALL
  SELECT to_port_id   AS port_id FROM routes WHERE registered = 0
),
port_activity AS (
  SELECT p.name, COUNT(*) AS unregistered_routes
  FROM unregistered u
  JOIN ports p ON p.id = u.port_id
  GROUP BY u.port_id, p.name
)
SELECT name, unregistered_routes
FROM port_activity
ORDER BY unregistered_routes DESC
LIMIT 5;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain why this optimized final query solves the mystery completely.

## Final Reflection

**Final answer format:** The port name, lowercase with underscore.

- State the final answer clearly.
- Explain why the evidence across all four sections supports that answer.
- Briefly note how correctness, efficiency, documentation, and creativity show up in the finished work.

---
*Generated by SQL:DUNGEON - CSCI 331 - Queens College CUNY*

# Mystery #10

## The Grand Conspiracy

**Track:** Kazi Individual SQLNOIR Mysteries  
**Quest ID:** Q010  
**Rank:** Legendary  
**Focus Areas:** Recursive CTE, Window Functions, NOT EXISTS, PIVOT, CASE WHEN, RANK  
**Exported:** 3/28/2026, 5:50:07 AM

*Narrative prompt*

The investigation has traced ten separate crimes back to a single coordinated network. Cells within cells. Money flowing in patterns only a window function can reveal. One name appears at every node. Find the architect.

**Track:** Kazi Individual SQLNOIR Mysteries  
**Quest ID:** Q010  
**Rank:** Legendary  
**Expected answer format:** The conspirator's name, lowercase.

## Sample Database Schema

### `conspirators`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |
| rank | TEXT |  |
| cell_id | INTEGER |  |

### `cells`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |
| leader_id | INTEGER |  |

### `wire_transfers`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| from_id | INTEGER |  |
| to_id | INTEGER |  |
| amount | INTEGER |  |
| tx_date | TEXT |  |
| flagged | INTEGER |  |

### `alibis`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| conspirator_id | INTEGER |  |
| alibi_date | TEXT |  |
| location | TEXT |  |
| confirmed | INTEGER |  |


## Section 1 - Part 1 — The Cell Hierarchy

**Section Goal:** The cells reference a leader_id that points back to a conspirator. Use a recursive CTE to traverse the chain of command from the top down.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [1]:
WITH RECURSIVE cell_hierarchy AS (
  SELECT
    c.id AS cell_id,
    c.name AS cell_name,
    c.leader_id,
    con.name AS leader_name,
    0 AS depth
  FROM cells c
  JOIN conspirators con ON con.id = c.leader_id
  WHERE c.leader_id = (
    SELECT MIN(leader_id) FROM cells
  )
  UNION ALL
  SELECT
    c2.id, c2.name, c2.leader_id,
    con2.name,
    ch.depth + 1
  FROM cells c2
  JOIN conspirators con2 ON con2.id = c2.leader_id
  JOIN cell_hierarchy ch  ON c2.leader_id = ch.leader_id
  WHERE c2.id != ch.cell_id
)
SELECT cell_name, leader_name, depth
FROM cell_hierarchy
ORDER BY depth, cell_name;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 2 - Part 2 — Follow the Money

**Section Goal:** Use a window function to sum each conspirator's total transfers sent and RANK them to find who moved the most money.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [2]:
WITH transfer_totals AS (
  SELECT
    con.name,
    SUM(wt.amount) AS total_sent,
    RANK() OVER (ORDER BY SUM(wt.amount) DESC) AS transfer_rank
  FROM wire_transfers wt
  JOIN conspirators con ON con.id = wt.from_id
  GROUP BY wt.from_id, con.name
)
SELECT name, total_sent, transfer_rank
FROM transfer_totals
ORDER BY transfer_rank;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 3 - Part 3 — No Alibi

**Section Goal:** Find conspirators who have NO confirmed alibi on any date when flagged wire transfers occurred.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [3]:
SELECT con.name
FROM conspirators con
WHERE NOT EXISTS (
  SELECT 1
  FROM alibis a
  JOIN wire_transfers wt
    ON wt.tx_date = a.alibi_date
   AND wt.flagged = 1
  WHERE a.conspirator_id = con.id
    AND a.confirmed = 1
)
ORDER BY con.name;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 4 - Part 4 — The Verdict

**Section Goal:** Assemble the grand CTE: cell hierarchy, transfer rank, no-alibi list, and monthly pivot. The architect is the conspirator who leads all cells, tops the money rank, and has no confirmed alibi.

This section should combine the prior evidence into one final, optimized answer.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [4]:
WITH transfer_totals AS (
  SELECT
    con.id AS con_id,
    con.name,
    SUM(wt.amount) AS total_sent,
    RANK() OVER (ORDER BY SUM(wt.amount) DESC) AS transfer_rank
  FROM wire_transfers wt
  JOIN conspirators con ON con.id = wt.from_id
  GROUP BY wt.from_id, con.name
),
no_alibi AS (
  SELECT con.id AS con_id, con.name
  FROM conspirators con
  WHERE NOT EXISTS (
    SELECT 1 FROM alibis a
    JOIN wire_transfers wt ON wt.tx_date = a.alibi_date AND wt.flagged = 1
    WHERE a.conspirator_id = con.id AND a.confirmed = 1
  )
),
monthly_pivot AS (
  SELECT
    from_id AS con_id,
    SUM(CASE WHEN tx_date LIKE '1247-01%' THEN amount ELSE 0 END) AS jan_sent,
    SUM(CASE WHEN tx_date LIKE '1247-02%' THEN amount ELSE 0 END) AS feb_sent,
    SUM(CASE WHEN tx_date LIKE '1247-03%' THEN amount ELSE 0 END) AS mar_sent
  FROM wire_transfers
  GROUP BY from_id
)
SELECT
  tt.name,
  tt.total_sent,
  tt.transfer_rank,
  mp.jan_sent,
  mp.feb_sent,
  mp.mar_sent
FROM transfer_totals tt
JOIN no_alibi     na ON na.con_id = tt.con_id
JOIN monthly_pivot mp ON mp.con_id = tt.con_id
WHERE tt.transfer_rank = 1
ORDER BY tt.total_sent DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain why this optimized final query solves the mystery completely.

## Final Reflection

**Final answer format:** The conspirator's name, lowercase.

- State the final answer clearly.
- Explain why the evidence across all four sections supports that answer.
- Briefly note how correctness, efficiency, documentation, and creativity show up in the finished work.

---
*Generated by SQL:DUNGEON - CSCI 331 - Queens College CUNY*

## Azm Individual SQLNOIR Mysteries

These 10 mysteries represent the second individual track, still following the same four-section SQLNOIR format.

# Mystery #1

## The Burn Rate Oracle

**Track:** Azm Individual SQLNOIR Mysteries  
**Quest ID:** AZ01  
**Rank:** Rare  
**Focus Areas:** Inventory, CTE, Aggregation, Forecasting  
**Exported:** 3/28/2026, 5:50:07 AM

*Narrative prompt*

Azm hands you a warehouse ledger instead of a mystery scroll. The answer is hidden in velocity, thresholds, and days of cover. Find the product that forces the fastest restock decision.

**Track:** Azm Individual SQLNOIR Mysteries  
**Quest ID:** AZ01  
**Rank:** Rare  
**Expected answer format:** Enter the product name in lowercase with underscore.

## Sample Database Schema

### `products`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |
| category | TEXT |  |
| unit_cost | INTEGER |  |
| unit_price | INTEGER |  |

### `stock`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| product_id | INTEGER |  |
| quantity_on_hand | INTEGER |  |
| restock_threshold | INTEGER |  |

### `sales_history`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| product_id | INTEGER |  |
| qty_sold | INTEGER |  |
| sale_date | TEXT |  |


## Section 1 - Situation Scan

**Section Goal:** Rank products by 30-day sales velocity as of 1247-03-15.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [1]:
SELECT p.name, SUM(sh.qty_sold) AS sold_30d
FROM sales_history sh
JOIN products p ON p.id = sh.product_id
WHERE sh.sale_date >= '1247-02-13'
GROUP BY sh.product_id, p.name
ORDER BY sold_30d DESC, p.name;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 2 - Pressure Point

**Section Goal:** Identify which products are currently below their restock threshold.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [2]:
SELECT p.name, s.quantity_on_hand, s.restock_threshold
FROM stock s
JOIN products p ON p.id = s.product_id
WHERE s.quantity_on_hand < s.restock_threshold
ORDER BY s.quantity_on_hand ASC, p.name;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 3 - Deep Cut

**Section Goal:** Estimate days of stock at the current 30-day velocity.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [3]:
WITH sold_30 AS (
  SELECT product_id, SUM(qty_sold) AS sold_30d
  FROM sales_history
  WHERE sale_date >= '1247-02-13'
  GROUP BY product_id
)
SELECT p.name, s.quantity_on_hand, COALESCE(s30.sold_30d, 0) AS sold_30d,
  ROUND(CASE WHEN COALESCE(s30.sold_30d, 0) = 0 THEN NULL
    ELSE s.quantity_on_hand / (s30.sold_30d / 30.0) END, 1) AS days_of_stock
FROM products p
JOIN stock s ON s.product_id = p.id
LEFT JOIN sold_30 s30 ON s30.product_id = p.id
ORDER BY days_of_stock ASC, p.name;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 4 - Final Recommendation

**Section Goal:** Choose the single product that should be reordered first.

This section should combine the prior evidence into one final, optimized answer.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [4]:
WITH sold_30 AS (
  SELECT product_id, SUM(qty_sold) AS sold_30d
  FROM sales_history
  WHERE sale_date >= '1247-02-13'
  GROUP BY product_id
),
priority AS (
  SELECT p.name, s.quantity_on_hand, s.restock_threshold, COALESCE(s30.sold_30d, 0) AS sold_30d,
    CASE WHEN COALESCE(s30.sold_30d, 0) = 0 THEN 9999
      ELSE ROUND(s.quantity_on_hand / (s30.sold_30d / 30.0), 1) END AS days_of_stock
  FROM products p
  JOIN stock s ON s.product_id = p.id
  LEFT JOIN sold_30 s30 ON s30.product_id = p.id
  WHERE s.quantity_on_hand < s.restock_threshold
)
SELECT name
FROM priority
ORDER BY days_of_stock ASC, sold_30d DESC, name
LIMIT 1;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain why this optimized final query solves the mystery completely.

## Final Reflection

**Final answer format:** Enter the product name in lowercase with underscore.

- State the final answer clearly.
- Explain why the evidence across all four sections supports that answer.
- Briefly note how correctness, efficiency, documentation, and creativity show up in the finished work.

---
*Generated by SQL:DUNGEON - CSCI 331 - Queens College CUNY*

# Mystery #2

## The Structuring Ledger

**Track:** Azm Individual SQLNOIR Mysteries  
**Quest ID:** AZ02  
**Rank:** Legendary  
**Focus Areas:** Fraud, CTE, Outliers, Aggregation  
**Exported:** 3/28/2026, 5:50:07 AM

*Narrative prompt*

Azm replaces monsters with money trails. One account hides in repetition instead of scale. Follow the account that keeps feeding the shadow merchant in nearly identical amounts.

**Track:** Azm Individual SQLNOIR Mysteries  
**Quest ID:** AZ02  
**Rank:** Legendary  
**Expected answer format:** Enter the account name in lowercase with underscore.

## Sample Database Schema

### `accounts`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |
| account_type | TEXT |  |

### `merchants`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |
| category | TEXT |  |

### `transactions`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| account_id | INTEGER |  |
| merchant_id | INTEGER |  |
| amount | INTEGER |  |
| tx_date | TEXT |  |


## Section 1 - Situation Scan

**Section Goal:** Measure the average transaction amount per account.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [1]:
SELECT a.name, AVG(t.amount) AS avg_amount, COUNT(*) AS tx_count
FROM transactions t
JOIN accounts a ON a.id = t.account_id
GROUP BY t.account_id, a.name
ORDER BY avg_amount DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 2 - Pressure Point

**Section Goal:** Flag transfers more than 2x the account average.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [2]:
WITH acct_avg AS (
  SELECT account_id, AVG(amount) AS avg_amount
  FROM transactions
  GROUP BY account_id
)
SELECT a.name, t.amount, t.tx_date, ROUND(t.amount / aa.avg_amount, 2) AS ratio
FROM transactions t
JOIN acct_avg aa ON aa.account_id = t.account_id
JOIN accounts a ON a.id = t.account_id
WHERE t.amount > aa.avg_amount * 2
ORDER BY ratio DESC, a.name;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 3 - Deep Cut

**Section Goal:** Show merchants receiving many small transactions that add up to suspicious volume.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [3]:
SELECT a.name, m.name AS merchant_name, COUNT(*) AS tx_count, SUM(t.amount) AS total_volume
FROM transactions t
JOIN accounts a ON a.id = t.account_id
JOIN merchants m ON m.id = t.merchant_id
GROUP BY t.account_id, a.name, m.name
ORDER BY tx_count DESC, total_volume DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 4 - Final Recommendation

**Section Goal:** Name the account with the clearest structuring pattern.

This section should combine the prior evidence into one final, optimized answer.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [4]:
SELECT a.name
FROM transactions t
JOIN accounts a ON a.id = t.account_id
JOIN merchants m ON m.id = t.merchant_id
WHERE m.name = 'shadow_exchange'
  AND t.amount BETWEEN 480 AND 500
GROUP BY a.id, a.name
HAVING COUNT(*) >= 5 AND SUM(t.amount) > 2000
ORDER BY SUM(t.amount) DESC, COUNT(*) DESC
LIMIT 1;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain why this optimized final query solves the mystery completely.

## Final Reflection

**Final answer format:** Enter the account name in lowercase with underscore.

- State the final answer clearly.
- Explain why the evidence across all four sections supports that answer.
- Briefly note how correctness, efficiency, documentation, and creativity show up in the finished work.

---
*Generated by SQL:DUNGEON - CSCI 331 - Queens College CUNY*

# Mystery #3

## The Coaching Writ

**Track:** Azm Individual SQLNOIR Mysteries  
**Quest ID:** AZ03  
**Rank:** Rare  
**Focus Areas:** Sales, RANK, Targets, Performance  
**Exported:** 3/28/2026, 5:50:07 AM

*Narrative prompt*

Azm does not want the flashiest rep. He wants the cleanest case for intervention. Find the salesperson whose attainment collapses under every lens you can measure.

**Track:** Azm Individual SQLNOIR Mysteries  
**Quest ID:** AZ03  
**Rank:** Rare  
**Expected answer format:** Enter the salesperson name in lowercase.

## Sample Database Schema

### `salespersons`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |
| region | TEXT |  |
| hire_date | TEXT |  |

### `sales`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| salesperson_id | INTEGER |  |
| amount | INTEGER |  |
| sale_date | TEXT |  |

### `targets`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| salesperson_id | INTEGER |  |
| month | TEXT |  |
| target_amount | INTEGER |  |


## Section 1 - Situation Scan

**Section Goal:** Rank the sales team by total sales.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [1]:
SELECT sp.name, sp.region, SUM(s.amount) AS total_sales
FROM sales s
JOIN salespersons sp ON sp.id = s.salesperson_id
GROUP BY sp.id, sp.name, sp.region
ORDER BY total_sales DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 2 - Pressure Point

**Section Goal:** Compute total target attainment for each salesperson.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [2]:
WITH actuals AS (
  SELECT salesperson_id, SUM(amount) AS total_actual
  FROM sales
  GROUP BY salesperson_id
),
targets_all AS (
  SELECT salesperson_id, SUM(target_amount) AS total_target
  FROM targets
  GROUP BY salesperson_id
)
SELECT sp.name, a.total_actual, t.total_target,
  ROUND(CAST(a.total_actual AS REAL) / t.total_target * 100, 1) AS attainment_pct
FROM salespersons sp
JOIN actuals a ON a.salesperson_id = sp.id
JOIN targets_all t ON t.salesperson_id = sp.id
ORDER BY attainment_pct DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 3 - Deep Cut

**Section Goal:** Rank salespeople within their region.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [3]:
WITH totals AS (
  SELECT sp.id, sp.name, sp.region, SUM(s.amount) AS total_sales
  FROM sales s
  JOIN salespersons sp ON sp.id = s.salesperson_id
  GROUP BY sp.id, sp.name, sp.region
)
SELECT name, region, total_sales,
  RANK() OVER (PARTITION BY region ORDER BY total_sales DESC) AS region_rank
FROM totals
ORDER BY region, region_rank;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 4 - Final Recommendation

**Section Goal:** Name the person who most clearly needs coaching.

This section should combine the prior evidence into one final, optimized answer.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [4]:
WITH actuals AS (
  SELECT salesperson_id, SUM(amount) AS total_actual
  FROM sales
  GROUP BY salesperson_id
),
targets_all AS (
  SELECT salesperson_id, SUM(target_amount) AS total_target
  FROM targets
  GROUP BY salesperson_id
)
SELECT sp.name
FROM salespersons sp
JOIN actuals a ON a.salesperson_id = sp.id
JOIN targets_all t ON t.salesperson_id = sp.id
ORDER BY CAST(a.total_actual AS REAL) / t.total_target ASC, sp.name
LIMIT 1;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain why this optimized final query solves the mystery completely.

## Final Reflection

**Final answer format:** Enter the salesperson name in lowercase.

- State the final answer clearly.
- Explain why the evidence across all four sections supports that answer.
- Briefly note how correctness, efficiency, documentation, and creativity show up in the finished work.

---
*Generated by SQL:DUNGEON - CSCI 331 - Queens College CUNY*

# Mystery #4

## The Recovery Ledger

**Track:** Azm Individual SQLNOIR Mysteries  
**Quest ID:** AZ04  
**Rank:** Legendary  
**Focus Areas:** Retention, Segmentation, Date Functions, LTV  
**Exported:** 3/28/2026, 5:50:07 AM

*Narrative prompt*

Azm does not chase every lost customer. He wants the one whose silence costs the guild the most. Find the churned member with the biggest value still worth recovering.

**Track:** Azm Individual SQLNOIR Mysteries  
**Quest ID:** AZ04  
**Rank:** Legendary  
**Expected answer format:** Enter the customer name in lowercase.

## Sample Database Schema

### `customers`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |
| tier | TEXT |  |

### `orders`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| customer_id | INTEGER |  |
| total_amount | INTEGER |  |
| order_date | TEXT |  |


## Section 1 - Situation Scan

**Section Goal:** Measure days since last order as of 1247-03-15.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [1]:
SELECT c.name, MAX(o.order_date) AS last_order,
  CAST(julianday('1247-03-15') - julianday(MAX(o.order_date)) AS INTEGER) AS days_since
FROM orders o
JOIN customers c ON c.id = o.customer_id
GROUP BY c.id, c.name
ORDER BY days_since ASC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 2 - Pressure Point

**Section Goal:** Segment each customer into Active, At Risk, or Churned.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [2]:
WITH recency AS (
  SELECT customer_id,
    CAST(julianday('1247-03-15') - julianday(MAX(order_date)) AS INTEGER) AS days_since
  FROM orders
  GROUP BY customer_id
)
SELECT c.name, r.days_since,
  CASE
    WHEN r.days_since < 30 THEN 'Active'
    WHEN r.days_since <= 90 THEN 'At Risk'
    ELSE 'Churned'
  END AS segment
FROM recency r
JOIN customers c ON c.id = r.customer_id
ORDER BY r.days_since DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 3 - Deep Cut

**Section Goal:** Compute lifetime value for each customer.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [3]:
SELECT c.name, c.tier, SUM(o.total_amount) AS lifetime_value
FROM orders o
JOIN customers c ON c.id = o.customer_id
GROUP BY c.id, c.name, c.tier
ORDER BY lifetime_value DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 4 - Final Recommendation

**Section Goal:** Name the best churned customer to re-engage first.

This section should combine the prior evidence into one final, optimized answer.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [4]:
WITH recency AS (
  SELECT customer_id,
    CAST(julianday('1247-03-15') - julianday(MAX(order_date)) AS INTEGER) AS days_since
  FROM orders
  GROUP BY customer_id
),
ltv AS (
  SELECT customer_id, SUM(total_amount) AS lifetime_value
  FROM orders
  GROUP BY customer_id
)
SELECT c.name
FROM customers c
JOIN recency r ON r.customer_id = c.id
JOIN ltv l ON l.customer_id = c.id
WHERE r.days_since > 90
ORDER BY l.lifetime_value DESC, c.name
LIMIT 1;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain why this optimized final query solves the mystery completely.

## Final Reflection

**Final answer format:** Enter the customer name in lowercase.

- State the final answer clearly.
- Explain why the evidence across all four sections supports that answer.
- Briefly note how correctness, efficiency, documentation, and creativity show up in the finished work.

---
*Generated by SQL:DUNGEON - CSCI 331 - Queens College CUNY*

# Mystery #5

## The Delay Gap

**Track:** Azm Individual SQLNOIR Mysteries  
**Quest ID:** AZ05  
**Rank:** Legendary  
**Focus Areas:** Supply Chain, Date Arithmetic, SLA, CTE  
**Exported:** 3/28/2026, 5:50:07 AM

*Narrative prompt*

Azm has no patience for vague blame. He wants the supplier whose promises collapse hardest once the full order life is measured end to end.

**Track:** Azm Individual SQLNOIR Mysteries  
**Quest ID:** AZ05  
**Rank:** Legendary  
**Expected answer format:** Enter the supplier name in lowercase with underscore.

## Sample Database Schema

### `suppliers`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |
| lead_time_days | INTEGER |  |

### `order_suppliers`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| order_id | INTEGER |  |
| supplier_id | INTEGER |  |

### `fulfillment_log`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| order_id | INTEGER |  |
| stage | TEXT |  |
| started_at | TEXT |  |
| completed_at | TEXT |  |


## Section 1 - Situation Scan

**Section Goal:** Measure average duration by stage.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [1]:
SELECT stage,
  ROUND(AVG(julianday(completed_at) - julianday(started_at)), 2) AS avg_days
FROM fulfillment_log
GROUP BY stage
ORDER BY avg_days DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 2 - Pressure Point

**Section Goal:** Measure total fulfillment days by order.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [2]:
SELECT order_id,
  ROUND(julianday(MAX(completed_at)) - julianday(MIN(started_at)), 1) AS total_days
FROM fulfillment_log
GROUP BY order_id
ORDER BY total_days DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 3 - Deep Cut

**Section Goal:** Compare actual order duration to each supplier promise.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [3]:
WITH order_duration AS (
  SELECT order_id,
    julianday(MAX(completed_at)) - julianday(MIN(started_at)) AS total_days
  FROM fulfillment_log
  GROUP BY order_id
)
SELECT s.name, s.lead_time_days,
  ROUND(AVG(od.total_days), 1) AS avg_actual_days,
  ROUND(AVG(od.total_days) - s.lead_time_days, 1) AS delay_gap
FROM order_suppliers os
JOIN suppliers s ON s.id = os.supplier_id
JOIN order_duration od ON od.order_id = os.order_id
GROUP BY s.id, s.name, s.lead_time_days
ORDER BY delay_gap DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 4 - Final Recommendation

**Section Goal:** Name the supplier creating the worst delay gap.

This section should combine the prior evidence into one final, optimized answer.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [4]:
WITH order_duration AS (
  SELECT order_id,
    julianday(MAX(completed_at)) - julianday(MIN(started_at)) AS total_days
  FROM fulfillment_log
  GROUP BY order_id
)
SELECT s.name
FROM order_suppliers os
JOIN suppliers s ON s.id = os.supplier_id
JOIN order_duration od ON od.order_id = os.order_id
GROUP BY s.id, s.name, s.lead_time_days
ORDER BY AVG(od.total_days) - s.lead_time_days DESC, s.name
LIMIT 1;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain why this optimized final query solves the mystery completely.

## Final Reflection

**Final answer format:** Enter the supplier name in lowercase with underscore.

- State the final answer clearly.
- Explain why the evidence across all four sections supports that answer.
- Briefly note how correctness, efficiency, documentation, and creativity show up in the finished work.

---
*Generated by SQL:DUNGEON - CSCI 331 - Queens College CUNY*

# Mystery #6

## The Margin Leak

**Track:** Azm Individual SQLNOIR Mysteries  
**Quest ID:** AZ06  
**Rank:** Epic  
**Focus Areas:** Pricing, Discounts, Margin, Aggregation  
**Exported:** 3/28/2026, 5:50:07 AM

*Narrative prompt*

Azm treats pricing drift like a breach in the wall. One bundle is leaking margin faster than the rest once approval limits and unit volume are combined.

**Track:** Azm Individual SQLNOIR Mysteries  
**Quest ID:** AZ06  
**Rank:** Epic  
**Expected answer format:** Enter the offer name in lowercase with underscore.

## Sample Database Schema

### `offers`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |
| list_price | INTEGER |  |
| current_price | INTEGER |  |

### `approvals`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| offer_id | INTEGER |  |
| approved_discount_pct | REAL |  |

### `reps`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |
| region | TEXT |  |

### `sales`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| offer_id | INTEGER |  |
| rep_id | INTEGER |  |
| units | INTEGER |  |
| sale_date | TEXT |  |


## Section 1 - Situation Scan

**Section Goal:** Calculate the actual discount percent for each offer.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [1]:
SELECT name, list_price, current_price,
  ROUND((list_price - current_price) * 100.0 / list_price, 1) AS actual_discount_pct
FROM offers
ORDER BY actual_discount_pct DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 2 - Pressure Point

**Section Goal:** Show which offers exceed their approved discount level.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [2]:
SELECT o.name,
  ROUND((o.list_price - o.current_price) * 100.0 / o.list_price, 1) AS actual_discount_pct,
  a.approved_discount_pct
FROM offers o
JOIN approvals a ON a.offer_id = o.id
WHERE ((o.list_price - o.current_price) * 100.0 / o.list_price) > a.approved_discount_pct
ORDER BY actual_discount_pct DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 3 - Deep Cut

**Section Goal:** Show discounted unit volume by rep.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [3]:
SELECT r.name, SUM(s.units) AS discounted_units
FROM sales s
JOIN reps r ON r.id = s.rep_id
GROUP BY r.id, r.name
ORDER BY discounted_units DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 4 - Final Recommendation

**Section Goal:** Name the offer leaking the most unauthorized revenue.

This section should combine the prior evidence into one final, optimized answer.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [4]:
WITH leak AS (
  SELECT o.name,
    ((o.list_price - o.current_price) * 100.0 / o.list_price) - a.approved_discount_pct AS extra_discount_pct,
    SUM(s.units) AS total_units
  FROM offers o
  JOIN approvals a ON a.offer_id = o.id
  JOIN sales s ON s.offer_id = o.id
  GROUP BY o.id, o.name, o.list_price, o.current_price, a.approved_discount_pct
)
SELECT name
FROM leak
ORDER BY extra_discount_pct * total_units DESC, name
LIMIT 1;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain why this optimized final query solves the mystery completely.

## Final Reflection

**Final answer format:** Enter the offer name in lowercase with underscore.

- State the final answer clearly.
- Explain why the evidence across all four sections supports that answer.
- Briefly note how correctness, efficiency, documentation, and creativity show up in the finished work.

---
*Generated by SQL:DUNGEON - CSCI 331 - Queens College CUNY*

# Mystery #7

## The Refund Spiral

**Track:** Azm Individual SQLNOIR Mysteries  
**Quest ID:** AZ07  
**Rank:** Rare  
**Focus Areas:** Returns, Customer Analysis, CASE WHEN, Aggregation  
**Exported:** 3/28/2026, 5:50:07 AM

*Narrative prompt*

Azm thinks one customer has learned to weaponize the return desk. Find the person whose return rate and damage-claim pressure rise together too often to be random.

**Track:** Azm Individual SQLNOIR Mysteries  
**Quest ID:** AZ07  
**Rank:** Rare  
**Expected answer format:** Enter the customer name in lowercase.

## Sample Database Schema

### `customers`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |

### `orders`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| customer_id | INTEGER |  |
| order_total | INTEGER |  |

### `returns`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| order_id | INTEGER |  |
| reason | TEXT |  |

### `tickets`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| customer_id | INTEGER |  |
| category | TEXT |  |


## Section 1 - Situation Scan

**Section Goal:** Count orders and returns by customer.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [1]:
SELECT c.name, COUNT(DISTINCT o.id) AS order_count, COUNT(r.id) AS return_count
FROM customers c
JOIN orders o ON o.customer_id = c.id
LEFT JOIN returns r ON r.order_id = o.id
GROUP BY c.id, c.name
ORDER BY return_count DESC, order_count DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 2 - Pressure Point

**Section Goal:** Measure return rate by customer.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [2]:
SELECT c.name,
  COUNT(r.id) * 1.0 / COUNT(DISTINCT o.id) AS return_rate
FROM customers c
JOIN orders o ON o.customer_id = c.id
LEFT JOIN returns r ON r.order_id = o.id
GROUP BY c.id, c.name
ORDER BY return_rate DESC, c.name;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 3 - Deep Cut

**Section Goal:** Count support tickets and damage-claim tickets by customer.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [3]:
SELECT c.name,
  COUNT(t.id) AS ticket_count,
  SUM(CASE WHEN t.category = 'damage_claim' THEN 1 ELSE 0 END) AS damage_claims
FROM customers c
LEFT JOIN tickets t ON t.customer_id = c.id
GROUP BY c.id, c.name
ORDER BY ticket_count DESC, damage_claims DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 4 - Final Recommendation

**Section Goal:** Name the customer whose behavior looks most abusive.

This section should combine the prior evidence into one final, optimized answer.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [4]:
WITH return_stats AS (
  SELECT c.id, c.name,
    COUNT(DISTINCT o.id) AS order_count,
    COUNT(r.id) AS return_count
  FROM customers c
  JOIN orders o ON o.customer_id = c.id
  LEFT JOIN returns r ON r.order_id = o.id
  GROUP BY c.id, c.name
),
ticket_stats AS (
  SELECT customer_id,
    COUNT(*) AS ticket_count,
    SUM(CASE WHEN category = 'damage_claim' THEN 1 ELSE 0 END) AS damage_claims
  FROM tickets
  GROUP BY customer_id
)
SELECT rs.name
FROM return_stats rs
JOIN ticket_stats ts ON ts.customer_id = rs.id
ORDER BY (rs.return_count * 1.0 / rs.order_count) DESC, ts.damage_claims DESC, ts.ticket_count DESC
LIMIT 1;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain why this optimized final query solves the mystery completely.

## Final Reflection

**Final answer format:** Enter the customer name in lowercase.

- State the final answer clearly.
- Explain why the evidence across all four sections supports that answer.
- Briefly note how correctness, efficiency, documentation, and creativity show up in the finished work.

---
*Generated by SQL:DUNGEON - CSCI 331 - Queens College CUNY*

# Mystery #8

## The Exhaustion Ledger

**Track:** Azm Individual SQLNOIR Mysteries  
**Quest ID:** AZ08  
**Rank:** Rare  
**Focus Areas:** Workforce, Overtime, Aggregation, Operations  
**Exported:** 3/28/2026, 5:50:07 AM

*Narrative prompt*

Azm sees payroll like a battlefield map. One team is soaking up more overtime than the rest, and the numbers have stopped being accidental.

**Track:** Azm Individual SQLNOIR Mysteries  
**Quest ID:** AZ08  
**Rank:** Rare  
**Expected answer format:** Enter the team name in lowercase with underscore.

## Sample Database Schema

### `teams`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |

### `employees`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |
| team_id | INTEGER |  |

### `schedules`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| employee_id | INTEGER |  |
| shift_hours | INTEGER |  |

### `clock_logs`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| employee_id | INTEGER |  |
| hours_worked | INTEGER |  |


## Section 1 - Situation Scan

**Section Goal:** Compare scheduled hours to actual hours by employee.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [1]:
SELECT e.name, t.name AS team_name, s.shift_hours, c.hours_worked,
  c.hours_worked - s.shift_hours AS overtime_hours
FROM employees e
JOIN teams t ON t.id = e.team_id
JOIN schedules s ON s.employee_id = e.id
JOIN clock_logs c ON c.employee_id = e.id
ORDER BY overtime_hours DESC, e.name;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 2 - Pressure Point

**Section Goal:** Sum overtime by employee.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [2]:
SELECT e.name, SUM(c.hours_worked - s.shift_hours) AS overtime_hours
FROM employees e
JOIN schedules s ON s.employee_id = e.id
JOIN clock_logs c ON c.employee_id = e.id
GROUP BY e.id, e.name
ORDER BY overtime_hours DESC, e.name;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 3 - Deep Cut

**Section Goal:** Sum overtime by team.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [3]:
SELECT t.name, SUM(c.hours_worked - s.shift_hours) AS overtime_hours
FROM teams t
JOIN employees e ON e.team_id = t.id
JOIN schedules s ON s.employee_id = e.id
JOIN clock_logs c ON c.employee_id = e.id
GROUP BY t.id, t.name
ORDER BY overtime_hours DESC, t.name;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 4 - Final Recommendation

**Section Goal:** Name the team carrying the heaviest overtime load.

This section should combine the prior evidence into one final, optimized answer.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [4]:
SELECT t.name
FROM teams t
JOIN employees e ON e.team_id = t.id
JOIN schedules s ON s.employee_id = e.id
JOIN clock_logs c ON c.employee_id = e.id
GROUP BY t.id, t.name
ORDER BY SUM(c.hours_worked - s.shift_hours) DESC, t.name
LIMIT 1;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain why this optimized final query solves the mystery completely.

## Final Reflection

**Final answer format:** Enter the team name in lowercase with underscore.

- State the final answer clearly.
- Explain why the evidence across all four sections supports that answer.
- Briefly note how correctness, efficiency, documentation, and creativity show up in the finished work.

---
*Generated by SQL:DUNGEON - CSCI 331 - Queens College CUNY*

# Mystery #9

## The Margin Atlas

**Track:** Azm Individual SQLNOIR Mysteries  
**Quest ID:** AZ09  
**Rank:** Epic  
**Focus Areas:** Logistics, Margin, CTE, Operations  
**Exported:** 3/28/2026, 5:50:07 AM

*Narrative prompt*

Azm cares less about noisy revenue than survivable lanes. Find the route whose margin collapses first once fuel, crew, and tolls are loaded against real shipment revenue.

**Track:** Azm Individual SQLNOIR Mysteries  
**Quest ID:** AZ09  
**Rank:** Epic  
**Expected answer format:** Enter the route name in lowercase with underscore.

## Sample Database Schema

### `routes`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |

### `shipments`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| route_id | INTEGER |  |
| revenue | INTEGER |  |

### `route_costs`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| route_id | INTEGER |  |
| fuel_cost | INTEGER |  |
| crew_cost | INTEGER |  |
| toll_cost | INTEGER |  |


## Section 1 - Situation Scan

**Section Goal:** Aggregate shipment revenue by route.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [1]:
SELECT r.name, SUM(s.revenue) AS total_revenue
FROM routes r
JOIN shipments s ON s.route_id = r.id
GROUP BY r.id, r.name
ORDER BY total_revenue DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 2 - Pressure Point

**Section Goal:** Aggregate total cost by route.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [2]:
SELECT r.name, rc.fuel_cost + rc.crew_cost + rc.toll_cost AS total_cost
FROM routes r
JOIN route_costs rc ON rc.route_id = r.id
ORDER BY total_cost DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 3 - Deep Cut

**Section Goal:** Calculate net margin by route.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [3]:
WITH revenue AS (
  SELECT route_id, SUM(revenue) AS total_revenue
  FROM shipments
  GROUP BY route_id
)
SELECT r.name, rev.total_revenue,
  rc.fuel_cost + rc.crew_cost + rc.toll_cost AS total_cost,
  rev.total_revenue - (rc.fuel_cost + rc.crew_cost + rc.toll_cost) AS net_margin
FROM routes r
JOIN revenue rev ON rev.route_id = r.id
JOIN route_costs rc ON rc.route_id = r.id
ORDER BY net_margin ASC, r.name;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 4 - Final Recommendation

**Section Goal:** Name the route that needs intervention first.

This section should combine the prior evidence into one final, optimized answer.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [4]:
WITH revenue AS (
  SELECT route_id, SUM(revenue) AS total_revenue
  FROM shipments
  GROUP BY route_id
)
SELECT r.name
FROM routes r
JOIN revenue rev ON rev.route_id = r.id
JOIN route_costs rc ON rc.route_id = r.id
ORDER BY rev.total_revenue - (rc.fuel_cost + rc.crew_cost + rc.toll_cost) ASC, r.name
LIMIT 1;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain why this optimized final query solves the mystery completely.

## Final Reflection

**Final answer format:** Enter the route name in lowercase with underscore.

- State the final answer clearly.
- Explain why the evidence across all four sections supports that answer.
- Briefly note how correctness, efficiency, documentation, and creativity show up in the finished work.

---
*Generated by SQL:DUNGEON - CSCI 331 - Queens College CUNY*

# Mystery #10

## The Supplier Reckoning

**Track:** Azm Individual SQLNOIR Mysteries  
**Quest ID:** AZ10  
**Rank:** Epic  
**Focus Areas:** Quality, Claims, Aggregation, Operations  
**Exported:** 3/28/2026, 5:50:07 AM

*Narrative prompt*

Azm wants the supplier whose defects keep echoing after receipt. Find the source whose lots create the worst combined burden of defects, rework, and claims.

**Track:** Azm Individual SQLNOIR Mysteries  
**Quest ID:** AZ10  
**Rank:** Epic  
**Expected answer format:** Enter the supplier name in lowercase with underscore.

## Sample Database Schema

### `suppliers`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |

### `lots`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| supplier_id | INTEGER |  |
| units_received | INTEGER |  |

### `inspections`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| lot_id | INTEGER |  |
| defects_found | INTEGER |  |
| rework_cost | INTEGER |  |

### `claims`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| lot_id | INTEGER |  |
| claim_amount | INTEGER |  |


## Section 1 - Situation Scan

**Section Goal:** Aggregate defects by supplier.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [1]:
SELECT s.name, SUM(i.defects_found) AS total_defects
FROM suppliers s
JOIN lots l ON l.supplier_id = s.id
JOIN inspections i ON i.lot_id = l.id
GROUP BY s.id, s.name
ORDER BY total_defects DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 2 - Pressure Point

**Section Goal:** Aggregate rework and claims cost by supplier.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [2]:
SELECT s.name,
  SUM(i.rework_cost) AS total_rework_cost,
  SUM(c.claim_amount) AS total_claim_cost
FROM suppliers s
JOIN lots l ON l.supplier_id = s.id
JOIN inspections i ON i.lot_id = l.id
JOIN claims c ON c.lot_id = l.id
GROUP BY s.id, s.name
ORDER BY total_rework_cost + total_claim_cost DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 3 - Deep Cut

**Section Goal:** Calculate a defect-adjusted cost signal by supplier.

This section should gather one important clue that feeds the final solution.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [3]:
SELECT s.name,
  SUM(i.defects_found) AS total_defects,
  SUM(i.rework_cost + c.claim_amount) AS total_quality_cost
FROM suppliers s
JOIN lots l ON l.supplier_id = s.id
JOIN inspections i ON i.lot_id = l.id
JOIN claims c ON c.lot_id = l.id
GROUP BY s.id, s.name
ORDER BY total_quality_cost DESC, total_defects DESC;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain what clue or hypothesis this section confirms.

## Section 4 - Final Recommendation

**Section Goal:** Name the supplier with the worst quality impact.

This section should combine the prior evidence into one final, optimized answer.

### Narrative Explanation

- What question are you answering in this section?
- Why does this step matter for the final mystery?
- What did you expect to learn before you ran the query?

In [4]:
SELECT s.name
FROM suppliers s
JOIN lots l ON l.supplier_id = s.id
JOIN inspections i ON i.lot_id = l.id
JOIN claims c ON c.lot_id = l.id
GROUP BY s.id, s.name
ORDER BY SUM(i.rework_cost + c.claim_amount) DESC, SUM(i.defects_found) DESC, s.name
LIMIT 1;

### Results and Interpretation

- Summarize the important rows, totals, rankings, or anomalies returned here.
- Explain how this result changes the next step in the investigation.
- Note any efficiency decisions, cleanup choices, or advanced SQL features used.
- Explain why this optimized final query solves the mystery completely.

## Final Reflection

**Final answer format:** Enter the supplier name in lowercase with underscore.

- State the final answer clearly.
- Explain why the evidence across all four sections supports that answer.
- Briefly note how correctness, efficiency, documentation, and creativity show up in the finished work.

---
*Generated by SQL:DUNGEON - CSCI 331 - Queens College CUNY*

## Guild Contracts - Paired Real-World Problem-Solving

These 10 sessions capture the navigator/driver collaboration work required for the paired portion of the project.

# Paired Real-World Problem-Solving

## Session 1: The Inventory Oracle

**Contract ID:** dc001  
**Lead Role This Session:** Navigator  
**Recommended Session Length:** 1-2 hours  
**Notebook Format:** Shared SQL notebook for navigator and driver collaboration

## Business Scenario

A guild warehouse is tying up too much cash. Some items are moving too fast, others are collecting dust, and the inventory team wants a restock and markdown plan before the next buying cycle.

## Role Structure

- Navigator acts as the business stakeholder and refines the requirement verbally.
- Driver acts as the programmer or analyst and writes the evolving T-SQL.
- Partners should switch emphasis midway if that improves balance or clarity.

- Navigator note: Define the business thresholds first: velocity, days of stock, and margin risk. Keep the Driver focused on which signals actually justify a restock recommendation.
- Driver note: Translate the thresholds into layered T-SQL with aggregates and a final prioritization query. Report back which products are urgent, slow, or margin-poor.

## Session Process

- Navigator states the business need, sharpens the question, and explains what a useful answer would change.
- Driver writes T-SQL in a shared notebook, tests results, and narrates what each iteration proves or disproves.
- Both partners refine the query path together, switch emphasis when needed, and keep role notes in the notebook.
- Finish with recommendations, query evolution, and a notebook trail strong enough for GitHub submission.

## Session Requirements

- Alternate navigator and driver responsibilities across sessions.
- Use advanced T-SQL patterns to move from raw data to a real recommendation.
- Capture role notes, query evolution, results, and a final recommendation in one shared notebook.
- Treat the work like a real business-technical conversation, not just a code dump.

## Contract-Specific Goals

- Rank products by 30-day sales velocity as of 1247-03-15.
- Identify which products are currently below their restock threshold.
- Estimate days of stock at the current 30-day velocity.
- Choose the single product that should be reordered first.

## Expected Deliverables

- A ranked restock priority list.
- A markdown shortlist for slow and overstocked items.
- A days-of-stock table with clear thresholds.
- A final recommendation naming the first item to reorder.

## Evaluation Criteria

- Collaboration: role alternation, listening, and requirement handoff stay clear throughout the session.
- Depth: the pair uses advanced T-SQL patterns such as CTEs, subqueries, ranking, pivots, or performance thinking.
- Innovation: the solution goes beyond a first-pass answer and explores edge cases, tuning, or richer analysis.
- Documentation: the notebook captures role notes, query evolution, evidence, and final recommendations.

## Sample Database Schema

### `products`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |
| category | TEXT |  |
| unit_cost | INTEGER |  |
| unit_price | INTEGER |  |

### `stock`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| product_id | INTEGER |  |
| quantity_on_hand | INTEGER |  |
| restock_threshold | INTEGER |  |

### `sales_history`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| product_id | INTEGER |  |
| qty_sold | INTEGER |  |
| sale_date | TEXT |  |


In [21]:
-- Session 1: The Inventory Oracle
-- Shared T-SQL workspace
-- Record the team's query evolution, tests, and refinements below.

-- Velocity Scan
-- Shows which products moved fastest in the latest 30-day window.
SELECT p.name, SUM(sh.qty_sold) AS sold_30d
FROM sales_history sh
JOIN products p ON p.id = sh.product_id
WHERE sh.sale_date >= '1247-02-13'
GROUP BY sh.product_id, p.name
ORDER BY sold_30d DESC, p.name;

-- Threshold Breach List
-- Surfaces the items already below their replenishment floor.
SELECT p.name, s.quantity_on_hand, s.restock_threshold
FROM stock s
JOIN products p ON p.id = s.product_id
WHERE s.quantity_on_hand < s.restock_threshold
ORDER BY s.quantity_on_hand ASC, p.name;

-- Days of Stock
-- Converts raw sales into a days-of-cover estimate.
WITH sold_30 AS (
  SELECT product_id, SUM(qty_sold) AS sold_30d
  FROM sales_history
  WHERE sale_date >= '1247-02-13'
  GROUP BY product_id
)
SELECT p.name, s.quantity_on_hand, COALESCE(s30.sold_30d, 0) AS sold_30d,
  ROUND(CASE WHEN COALESCE(s30.sold_30d, 0) = 0 THEN NULL
    ELSE s.quantity_on_hand / (s30.sold_30d / 30.0) END, 1) AS days_of_stock
FROM products p
JOIN stock s ON s.product_id = p.id
LEFT JOIN sold_30 s30 ON s30.product_id = p.id
ORDER BY days_of_stock ASC, p.name;

-- Add the pair's evolving T-SQL here

## Pair Notes and Recommendation

### Navigator Notes
- What business need or requirement was clarified during the session?
- Which edge cases or decision points mattered most?

### Driver Notes
- How did the query evolve from first attempt to final answer?
- Which advanced T-SQL features helped most here?

### Final Recommendation
- Summarize the insight, action item, or business recommendation.
- Note any performance tuning, validation checks, or optional visualization follow-up.

# Paired Real-World Problem-Solving

## Session 2: The Fraud Tribunal

**Contract ID:** dc002  
**Lead Role This Session:** Driver  
**Recommended Session Length:** 1-2 hours  
**Notebook Format:** Shared SQL notebook for navigator and driver collaboration

## Business Scenario

Bank auditors suspect structuring: many small transfers to the same shadow merchant instead of a few obvious large ones. The pair needs a defensible fraud shortlist before compliance locks the accounts.

## Role Structure

- Navigator acts as the business stakeholder and refines the requirement verbally.
- Driver acts as the programmer or analyst and writes the evolving T-SQL.
- Partners should switch emphasis midway if that improves balance or clarity.

- Navigator note: Set the suspicion rules clearly: outlier behavior against account norms, repeated small transfers, and concentration into a risky merchant.
- Driver note: Build the account-average baseline, the merchant concentration view, and the final structuring filter in T-SQL.

## Session Process

- Navigator states the business need, sharpens the question, and explains what a useful answer would change.
- Driver writes T-SQL in a shared notebook, tests results, and narrates what each iteration proves or disproves.
- Both partners refine the query path together, switch emphasis when needed, and keep role notes in the notebook.
- Finish with recommendations, query evolution, and a notebook trail strong enough for GitHub submission.

## Session Requirements

- Alternate navigator and driver responsibilities across sessions.
- Use advanced T-SQL patterns to move from raw data to a real recommendation.
- Capture role notes, query evolution, results, and a final recommendation in one shared notebook.
- Treat the work like a real business-technical conversation, not just a code dump.

## Contract-Specific Goals

- Measure the average transaction amount per account.
- Flag transfers more than 2x the account average.
- Show merchants receiving many small transactions that add up to suspicious volume.
- Name the account with the clearest structuring pattern.

## Expected Deliverables

- A table of outlier transfers with account context.
- A merchant concentration report focused on small repeated transfers.
- A shortlist of suspicious accounts.
- A final written fraud verdict naming the strongest structuring candidate.

## Evaluation Criteria

- Collaboration: role alternation, listening, and requirement handoff stay clear throughout the session.
- Depth: the pair uses advanced T-SQL patterns such as CTEs, subqueries, ranking, pivots, or performance thinking.
- Innovation: the solution goes beyond a first-pass answer and explores edge cases, tuning, or richer analysis.
- Documentation: the notebook captures role notes, query evolution, evidence, and final recommendations.

## Sample Database Schema

### `accounts`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |
| account_type | TEXT |  |

### `merchants`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |
| category | TEXT |  |

### `transactions`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| account_id | INTEGER |  |
| merchant_id | INTEGER |  |
| amount | INTEGER |  |
| tx_date | TEXT |  |


In [22]:
-- Session 2: The Fraud Tribunal
-- Shared T-SQL workspace
-- Record the team's query evolution, tests, and refinements below.

-- Account Average Baseline
-- Builds the normal transaction profile for each account.
SELECT a.name, AVG(t.amount) AS avg_amount, COUNT(*) AS tx_count
FROM transactions t
JOIN accounts a ON a.id = t.account_id
GROUP BY t.account_id, a.name
ORDER BY avg_amount DESC;

-- 2x Outlier Transfers
-- Flags transactions that break their own account norm.
WITH acct_avg AS (
  SELECT account_id, AVG(amount) AS avg_amount
  FROM transactions
  GROUP BY account_id
)
SELECT a.name, t.amount, t.tx_date, ROUND(t.amount / aa.avg_amount, 2) AS ratio
FROM transactions t
JOIN acct_avg aa ON aa.account_id = t.account_id
JOIN accounts a ON a.id = t.account_id
WHERE t.amount > aa.avg_amount * 2
ORDER BY ratio DESC, a.name;

-- Merchant Concentration
-- Shows repeated payment behavior by account and merchant.
SELECT a.name, m.name AS merchant_name, COUNT(*) AS tx_count, SUM(t.amount) AS total_volume
FROM transactions t
JOIN accounts a ON a.id = t.account_id
JOIN merchants m ON m.id = t.merchant_id
GROUP BY t.account_id, a.name, m.name
ORDER BY tx_count DESC, total_volume DESC;

-- Add the pair's evolving T-SQL here

## Pair Notes and Recommendation

### Navigator Notes
- What business need or requirement was clarified during the session?
- Which edge cases or decision points mattered most?

### Driver Notes
- How did the query evolve from first attempt to final answer?
- Which advanced T-SQL features helped most here?

### Final Recommendation
- Summarize the insight, action item, or business recommendation.
- Note any performance tuning, validation checks, or optional visualization follow-up.

# Paired Real-World Problem-Solving

## Session 3: The Sales Tribunal

**Contract ID:** dc003  
**Lead Role This Session:** Navigator  
**Recommended Session Length:** 1-2 hours  
**Notebook Format:** Shared SQL notebook for navigator and driver collaboration

## Business Scenario

Regional sales results are uneven, management wants coaching assignments, and nobody agrees on whether tenure or territory is the real issue. The pair must turn raw sales into a fair performance story.

## Role Structure

- Navigator acts as the business stakeholder and refines the requirement verbally.
- Driver acts as the programmer or analyst and writes the evolving T-SQL.
- Partners should switch emphasis midway if that improves balance or clarity.

- Navigator note: Define the evaluation order: totals first, attainment next, then regional context. Keep the analysis business-facing, not just leaderboard-heavy.
- Driver note: Use aggregates, joins, and ranking functions to surface total sales, target attainment, and within-region rank.

## Session Process

- Navigator states the business need, sharpens the question, and explains what a useful answer would change.
- Driver writes T-SQL in a shared notebook, tests results, and narrates what each iteration proves or disproves.
- Both partners refine the query path together, switch emphasis when needed, and keep role notes in the notebook.
- Finish with recommendations, query evolution, and a notebook trail strong enough for GitHub submission.

## Session Requirements

- Alternate navigator and driver responsibilities across sessions.
- Use advanced T-SQL patterns to move from raw data to a real recommendation.
- Capture role notes, query evolution, results, and a final recommendation in one shared notebook.
- Treat the work like a real business-technical conversation, not just a code dump.

## Contract-Specific Goals

- Rank the sales team by total sales.
- Compute total target attainment for each salesperson.
- Rank salespeople within their region.
- Name the person who most clearly needs coaching.

## Expected Deliverables

- A team leaderboard by total sales.
- A target attainment table.
- A regional rank summary.
- A final coaching recommendation naming the weakest performer.

## Evaluation Criteria

- Collaboration: role alternation, listening, and requirement handoff stay clear throughout the session.
- Depth: the pair uses advanced T-SQL patterns such as CTEs, subqueries, ranking, pivots, or performance thinking.
- Innovation: the solution goes beyond a first-pass answer and explores edge cases, tuning, or richer analysis.
- Documentation: the notebook captures role notes, query evolution, evidence, and final recommendations.

## Sample Database Schema

### `salespersons`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |
| region | TEXT |  |
| hire_date | TEXT |  |

### `sales`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| salesperson_id | INTEGER |  |
| amount | INTEGER |  |
| sale_date | TEXT |  |

### `targets`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| salesperson_id | INTEGER |  |
| month | TEXT |  |
| target_amount | INTEGER |  |


In [23]:
-- Session 3: The Sales Tribunal
-- Shared T-SQL workspace
-- Record the team's query evolution, tests, and refinements below.

-- Total Sales
-- Builds the raw performance leaderboard.
SELECT sp.name, sp.region, SUM(s.amount) AS total_sales
FROM sales s
JOIN salespersons sp ON sp.id = s.salesperson_id
GROUP BY sp.id, sp.name, sp.region
ORDER BY total_sales DESC;

-- Target Attainment
-- Shows who is converting effort into target hit rate.
WITH actuals AS (
  SELECT salesperson_id, SUM(amount) AS total_actual
  FROM sales
  GROUP BY salesperson_id
),
targets_all AS (
  SELECT salesperson_id, SUM(target_amount) AS total_target
  FROM targets
  GROUP BY salesperson_id
)
SELECT sp.name, a.total_actual, t.total_target,
  ROUND(CAST(a.total_actual AS REAL) / t.total_target * 100, 1) AS attainment_pct
FROM salespersons sp
JOIN actuals a ON a.salesperson_id = sp.id
JOIN targets_all t ON t.salesperson_id = sp.id
ORDER BY attainment_pct DESC;

-- Regional Rank
-- Adds context so weak performance is not judged without territory comparison.
WITH totals AS (
  SELECT sp.id, sp.name, sp.region, SUM(s.amount) AS total_sales
  FROM sales s
  JOIN salespersons sp ON sp.id = s.salesperson_id
  GROUP BY sp.id, sp.name, sp.region
)
SELECT name, region, total_sales,
  RANK() OVER (PARTITION BY region ORDER BY total_sales DESC) AS region_rank
FROM totals
ORDER BY region, region_rank;

-- Add the pair's evolving T-SQL here

## Pair Notes and Recommendation

### Navigator Notes
- What business need or requirement was clarified during the session?
- Which edge cases or decision points mattered most?

### Driver Notes
- How did the query evolve from first attempt to final answer?
- Which advanced T-SQL features helped most here?

### Final Recommendation
- Summarize the insight, action item, or business recommendation.
- Note any performance tuning, validation checks, or optional visualization follow-up.

# Paired Real-World Problem-Solving

## Session 4: The Loyalty Index

**Contract ID:** dc004  
**Lead Role This Session:** Driver  
**Recommended Session Length:** 1-2 hours  
**Notebook Format:** Shared SQL notebook for navigator and driver collaboration

## Business Scenario

A rival network is winning back old customers. The pair must segment churn risk, estimate value at stake, and decide which dormant members are worth re-engaging first.

## Role Structure

- Navigator acts as the business stakeholder and refines the requirement verbally.
- Driver acts as the programmer or analyst and writes the evolving T-SQL.
- Partners should switch emphasis midway if that improves balance or clarity.

- Navigator note: Define the recency buckets and keep the Driver focused on business value, not just inactivity.
- Driver note: Use date arithmetic, aggregation, and segmentation logic to turn order history into a retention plan.

## Session Process

- Navigator states the business need, sharpens the question, and explains what a useful answer would change.
- Driver writes T-SQL in a shared notebook, tests results, and narrates what each iteration proves or disproves.
- Both partners refine the query path together, switch emphasis when needed, and keep role notes in the notebook.
- Finish with recommendations, query evolution, and a notebook trail strong enough for GitHub submission.

## Session Requirements

- Alternate navigator and driver responsibilities across sessions.
- Use advanced T-SQL patterns to move from raw data to a real recommendation.
- Capture role notes, query evolution, results, and a final recommendation in one shared notebook.
- Treat the work like a real business-technical conversation, not just a code dump.

## Contract-Specific Goals

- Measure days since last order as of 1247-03-15.
- Segment each customer into Active, At Risk, or Churned.
- Compute lifetime value for each customer.
- Name the best churned customer to re-engage first.

## Expected Deliverables

- A recency and segment table for all customers.
- A lifetime value ranking.
- A re-engagement shortlist for churned members.
- A final recommendation naming the top re-engagement target.

## Evaluation Criteria

- Collaboration: role alternation, listening, and requirement handoff stay clear throughout the session.
- Depth: the pair uses advanced T-SQL patterns such as CTEs, subqueries, ranking, pivots, or performance thinking.
- Innovation: the solution goes beyond a first-pass answer and explores edge cases, tuning, or richer analysis.
- Documentation: the notebook captures role notes, query evolution, evidence, and final recommendations.

## Sample Database Schema

### `customers`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |
| tier | TEXT |  |

### `orders`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| customer_id | INTEGER |  |
| total_amount | INTEGER |  |
| order_date | TEXT |  |


In [24]:
-- Session 4: The Loyalty Index
-- Shared T-SQL workspace
-- Record the team's query evolution, tests, and refinements below.

-- Recency Scan
-- Shows how long each customer has been idle.
SELECT c.name, MAX(o.order_date) AS last_order,
  CAST(julianday('1247-03-15') - julianday(MAX(o.order_date)) AS INTEGER) AS days_since
FROM orders o
JOIN customers c ON c.id = o.customer_id
GROUP BY c.id, c.name
ORDER BY days_since ASC;

-- Churn Segmentation
-- Turns recency into an actionable retention bucket.
WITH recency AS (
  SELECT customer_id,
    CAST(julianday('1247-03-15') - julianday(MAX(order_date)) AS INTEGER) AS days_since
  FROM orders
  GROUP BY customer_id
)
SELECT c.name, r.days_since,
  CASE
    WHEN r.days_since < 30 THEN 'Active'
    WHEN r.days_since <= 90 THEN 'At Risk'
    ELSE 'Churned'
  END AS segment
FROM recency r
JOIN customers c ON c.id = r.customer_id
ORDER BY r.days_since DESC;

-- Lifetime Value
-- Adds value so the team can prioritize recovery effort.
SELECT c.name, c.tier, SUM(o.total_amount) AS lifetime_value
FROM orders o
JOIN customers c ON c.id = o.customer_id
GROUP BY c.id, c.name, c.tier
ORDER BY lifetime_value DESC;

-- Add the pair's evolving T-SQL here

## Pair Notes and Recommendation

### Navigator Notes
- What business need or requirement was clarified during the session?
- Which edge cases or decision points mattered most?

### Driver Notes
- How did the query evolve from first attempt to final answer?
- Which advanced T-SQL features helped most here?

### Final Recommendation
- Summarize the insight, action item, or business recommendation.
- Note any performance tuning, validation checks, or optional visualization follow-up.

# Paired Real-World Problem-Solving

## Session 5: The Chain of Blame

**Contract ID:** dc005  
**Lead Role This Session:** Navigator  
**Recommended Session Length:** 1-2 hours  
**Notebook Format:** Shared SQL notebook for navigator and driver collaboration

## Business Scenario

Late shipments are setting off internal blame wars. The pair must identify the worst stage and the supplier most responsible for missed SLAs before leadership punishes the wrong team.

## Role Structure

- Navigator acts as the business stakeholder and refines the requirement verbally.
- Driver acts as the programmer or analyst and writes the evolving T-SQL.
- Partners should switch emphasis midway if that improves balance or clarity.

- Navigator note: Choose whether to judge delay by stage averages, supplier gap, or both. Keep the final story focused on root cause, not just noisy delays.
- Driver note: Use date arithmetic and grouped comparisons to measure stage duration, order duration, and supplier delay gap.

## Session Process

- Navigator states the business need, sharpens the question, and explains what a useful answer would change.
- Driver writes T-SQL in a shared notebook, tests results, and narrates what each iteration proves or disproves.
- Both partners refine the query path together, switch emphasis when needed, and keep role notes in the notebook.
- Finish with recommendations, query evolution, and a notebook trail strong enough for GitHub submission.

## Session Requirements

- Alternate navigator and driver responsibilities across sessions.
- Use advanced T-SQL patterns to move from raw data to a real recommendation.
- Capture role notes, query evolution, results, and a final recommendation in one shared notebook.
- Treat the work like a real business-technical conversation, not just a code dump.

## Contract-Specific Goals

- Measure average duration by stage.
- Measure total fulfillment days by order.
- Compare actual order duration to each supplier promise.
- Name the supplier creating the worst delay gap.

## Expected Deliverables

- A stage duration summary.
- An SLA breach list by order.
- A supplier gap report.
- A root-cause verdict naming the worst supplier.

## Evaluation Criteria

- Collaboration: role alternation, listening, and requirement handoff stay clear throughout the session.
- Depth: the pair uses advanced T-SQL patterns such as CTEs, subqueries, ranking, pivots, or performance thinking.
- Innovation: the solution goes beyond a first-pass answer and explores edge cases, tuning, or richer analysis.
- Documentation: the notebook captures role notes, query evolution, evidence, and final recommendations.

## Sample Database Schema

### `suppliers`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |
| lead_time_days | INTEGER |  |

### `order_suppliers`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| order_id | INTEGER |  |
| supplier_id | INTEGER |  |

### `fulfillment_log`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| order_id | INTEGER |  |
| stage | TEXT |  |
| started_at | TEXT |  |
| completed_at | TEXT |  |


In [25]:
-- Session 5: The Chain of Blame
-- Shared T-SQL workspace
-- Record the team's query evolution, tests, and refinements below.

-- Stage Duration
-- Shows which stage is slowest on average.
SELECT stage,
  ROUND(AVG(julianday(completed_at) - julianday(started_at)), 2) AS avg_days
FROM fulfillment_log
GROUP BY stage
ORDER BY avg_days DESC;

-- Order Duration
-- Shows which orders actually breached expectations.
SELECT order_id,
  ROUND(julianday(MAX(completed_at)) - julianday(MIN(started_at)), 1) AS total_days
FROM fulfillment_log
GROUP BY order_id
ORDER BY total_days DESC;

-- Supplier Delay Gap
-- Shows which supplier promise diverges most from reality.
WITH order_duration AS (
  SELECT order_id,
    julianday(MAX(completed_at)) - julianday(MIN(started_at)) AS total_days
  FROM fulfillment_log
  GROUP BY order_id
)
SELECT s.name, s.lead_time_days,
  ROUND(AVG(od.total_days), 1) AS avg_actual_days,
  ROUND(AVG(od.total_days) - s.lead_time_days, 1) AS delay_gap
FROM order_suppliers os
JOIN suppliers s ON s.id = os.supplier_id
JOIN order_duration od ON od.order_id = os.order_id
GROUP BY s.id, s.name, s.lead_time_days
ORDER BY delay_gap DESC;

-- Add the pair's evolving T-SQL here

## Pair Notes and Recommendation

### Navigator Notes
- What business need or requirement was clarified during the session?
- Which edge cases or decision points mattered most?

### Driver Notes
- How did the query evolve from first attempt to final answer?
- Which advanced T-SQL features helped most here?

### Final Recommendation
- Summarize the insight, action item, or business recommendation.
- Note any performance tuning, validation checks, or optional visualization follow-up.

# Paired Real-World Problem-Solving

## Session 6: The Discount Leak Ledger

**Contract ID:** dc006  
**Lead Role This Session:** Driver  
**Recommended Session Length:** 1-2 hours  
**Notebook Format:** Shared SQL notebook for navigator and driver collaboration

## Business Scenario

Discount approvals are being ignored in the field. The pair must identify which bundle is leaking the most margin and which reps keep selling beyond approved markdown levels.

## Role Structure

- Navigator acts as the business stakeholder and refines the requirement verbally.
- Driver acts as the programmer or analyst and writes the evolving T-SQL.
- Partners should switch emphasis midway if that improves balance or clarity.

- Navigator note: Define what counts as abuse: actual discount above approved discount, plus enough unit volume to matter.
- Driver note: Use price math, grouped sales totals, and a final leak calculation to name the offer creating the biggest unauthorized loss.

## Session Process

- Navigator states the business need, sharpens the question, and explains what a useful answer would change.
- Driver writes T-SQL in a shared notebook, tests results, and narrates what each iteration proves or disproves.
- Both partners refine the query path together, switch emphasis when needed, and keep role notes in the notebook.
- Finish with recommendations, query evolution, and a notebook trail strong enough for GitHub submission.

## Session Requirements

- Alternate navigator and driver responsibilities across sessions.
- Use advanced T-SQL patterns to move from raw data to a real recommendation.
- Capture role notes, query evolution, results, and a final recommendation in one shared notebook.
- Treat the work like a real business-technical conversation, not just a code dump.

## Contract-Specific Goals

- Calculate the actual discount percent for each offer.
- Show which offers exceed their approved discount level.
- Show discounted unit volume by rep.
- Name the offer leaking the most unauthorized revenue.

## Expected Deliverables

- An actual-vs-approved discount table.
- A rep volume view.
- An unauthorized discount shortlist.
- A final answer naming the worst margin leak.

## Evaluation Criteria

- Collaboration: role alternation, listening, and requirement handoff stay clear throughout the session.
- Depth: the pair uses advanced T-SQL patterns such as CTEs, subqueries, ranking, pivots, or performance thinking.
- Innovation: the solution goes beyond a first-pass answer and explores edge cases, tuning, or richer analysis.
- Documentation: the notebook captures role notes, query evolution, evidence, and final recommendations.

## Sample Database Schema

### `offers`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |
| list_price | INTEGER |  |
| current_price | INTEGER |  |

### `approvals`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| offer_id | INTEGER |  |
| approved_discount_pct | REAL |  |

### `reps`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |
| region | TEXT |  |

### `sales`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| offer_id | INTEGER |  |
| rep_id | INTEGER |  |
| units | INTEGER |  |
| sale_date | TEXT |  |


In [26]:
-- Session 6: The Discount Leak Ledger
-- Shared T-SQL workspace
-- Record the team's query evolution, tests, and refinements below.

-- Actual Discount Table
-- Shows the markdown actually being given in the field.
SELECT name, list_price, current_price,
  ROUND((list_price - current_price) * 100.0 / list_price, 1) AS actual_discount_pct
FROM offers
ORDER BY actual_discount_pct DESC;

-- Approval Breaches
-- Shows which offers are discounting beyond approval.
SELECT o.name,
  ROUND((o.list_price - o.current_price) * 100.0 / o.list_price, 1) AS actual_discount_pct,
  a.approved_discount_pct
FROM offers o
JOIN approvals a ON a.offer_id = o.id
WHERE ((o.list_price - o.current_price) * 100.0 / o.list_price) > a.approved_discount_pct
ORDER BY actual_discount_pct DESC;

-- Rep Discount Volume
-- Shows who is moving the most discounted volume.
SELECT r.name, SUM(s.units) AS discounted_units
FROM sales s
JOIN reps r ON r.id = s.rep_id
GROUP BY r.id, r.name
ORDER BY discounted_units DESC;

-- Add the pair's evolving T-SQL here

## Pair Notes and Recommendation

### Navigator Notes
- What business need or requirement was clarified during the session?
- Which edge cases or decision points mattered most?

### Driver Notes
- How did the query evolve from first attempt to final answer?
- Which advanced T-SQL features helped most here?

### Final Recommendation
- Summarize the insight, action item, or business recommendation.
- Note any performance tuning, validation checks, or optional visualization follow-up.

# Paired Real-World Problem-Solving

## Session 7: The Return Rift

**Contract ID:** dc007  
**Lead Role This Session:** Navigator  
**Recommended Session Length:** 1-2 hours  
**Notebook Format:** Shared SQL notebook for navigator and driver collaboration

## Business Scenario

Returns are climbing, but only a few customers are driving most of the cost. The pair must separate normal returns from abuse by looking at repeat patterns, reason codes, and support escalation volume.

## Role Structure

- Navigator acts as the business stakeholder and refines the requirement verbally.
- Driver acts as the programmer or analyst and writes the evolving T-SQL.
- Partners should switch emphasis midway if that improves balance or clarity.

- Navigator note: Define a suspicious pattern before the Driver starts coding: high return rate, repeated damage reasons, and heavy ticket volume.
- Driver note: Use grouped comparisons and joined evidence to calculate return rate and identify the strongest abuse signal.

## Session Process

- Navigator states the business need, sharpens the question, and explains what a useful answer would change.
- Driver writes T-SQL in a shared notebook, tests results, and narrates what each iteration proves or disproves.
- Both partners refine the query path together, switch emphasis when needed, and keep role notes in the notebook.
- Finish with recommendations, query evolution, and a notebook trail strong enough for GitHub submission.

## Session Requirements

- Alternate navigator and driver responsibilities across sessions.
- Use advanced T-SQL patterns to move from raw data to a real recommendation.
- Capture role notes, query evolution, results, and a final recommendation in one shared notebook.
- Treat the work like a real business-technical conversation, not just a code dump.

## Contract-Specific Goals

- Count orders and returns by customer.
- Measure return rate by customer.
- Count support tickets and damage-claim tickets by customer.
- Name the customer whose behavior looks most abusive.

## Expected Deliverables

- An order vs return table.
- A customer return-rate table.
- A support escalation summary.
- A final abuse verdict naming the strongest suspect.

## Evaluation Criteria

- Collaboration: role alternation, listening, and requirement handoff stay clear throughout the session.
- Depth: the pair uses advanced T-SQL patterns such as CTEs, subqueries, ranking, pivots, or performance thinking.
- Innovation: the solution goes beyond a first-pass answer and explores edge cases, tuning, or richer analysis.
- Documentation: the notebook captures role notes, query evolution, evidence, and final recommendations.

## Sample Database Schema

### `customers`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |

### `orders`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| customer_id | INTEGER |  |
| order_total | INTEGER |  |

### `returns`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| order_id | INTEGER |  |
| reason | TEXT |  |

### `tickets`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| customer_id | INTEGER |  |
| category | TEXT |  |


In [27]:
-- Session 7: The Return Rift
-- Shared T-SQL workspace
-- Record the team's query evolution, tests, and refinements below.

-- Orders vs Returns
-- Shows who returns the largest share of what they order.
SELECT c.name, COUNT(DISTINCT o.id) AS order_count, COUNT(r.id) AS return_count
FROM customers c
JOIN orders o ON o.customer_id = c.id
LEFT JOIN returns r ON r.order_id = o.id
GROUP BY c.id, c.name
ORDER BY return_count DESC, order_count DESC;

-- Return Rate
-- Converts volume into rate so small customers do not hide behind low counts.
SELECT c.name,
  COUNT(r.id) * 1.0 / COUNT(DISTINCT o.id) AS return_rate
FROM customers c
JOIN orders o ON o.customer_id = c.id
LEFT JOIN returns r ON r.order_id = o.id
GROUP BY c.id, c.name
ORDER BY return_rate DESC, c.name;

-- Ticket Pressure
-- Adds support burden to the return story.
SELECT c.name,
  COUNT(t.id) AS ticket_count,
  SUM(CASE WHEN t.category = 'damage_claim' THEN 1 ELSE 0 END) AS damage_claims
FROM customers c
LEFT JOIN tickets t ON t.customer_id = c.id
GROUP BY c.id, c.name
ORDER BY ticket_count DESC, damage_claims DESC;

-- Add the pair's evolving T-SQL here

## Pair Notes and Recommendation

### Navigator Notes
- What business need or requirement was clarified during the session?
- Which edge cases or decision points mattered most?

### Driver Notes
- How did the query evolve from first attempt to final answer?
- Which advanced T-SQL features helped most here?

### Final Recommendation
- Summarize the insight, action item, or business recommendation.
- Note any performance tuning, validation checks, or optional visualization follow-up.

# Paired Real-World Problem-Solving

## Session 8: The Overtime Apparition

**Contract ID:** dc008  
**Lead Role This Session:** Driver  
**Recommended Session Length:** 1-2 hours  
**Notebook Format:** Shared SQL notebook for navigator and driver collaboration

## Business Scenario

Labor costs spiked after schedules were tightened. Leadership wants to know whether one team is absorbing too much overtime or whether the issue is spread across the floor.

## Role Structure

- Navigator acts as the business stakeholder and refines the requirement verbally.
- Driver acts as the programmer or analyst and writes the evolving T-SQL.
- Partners should switch emphasis midway if that improves balance or clarity.

- Navigator note: Decide whether the story should focus on teams, people, or both. Keep the analysis tied to staffing decisions.
- Driver note: Compare scheduled hours to logged hours, aggregate overtime, and surface the team driving the largest overage.

## Session Process

- Navigator states the business need, sharpens the question, and explains what a useful answer would change.
- Driver writes T-SQL in a shared notebook, tests results, and narrates what each iteration proves or disproves.
- Both partners refine the query path together, switch emphasis when needed, and keep role notes in the notebook.
- Finish with recommendations, query evolution, and a notebook trail strong enough for GitHub submission.

## Session Requirements

- Alternate navigator and driver responsibilities across sessions.
- Use advanced T-SQL patterns to move from raw data to a real recommendation.
- Capture role notes, query evolution, results, and a final recommendation in one shared notebook.
- Treat the work like a real business-technical conversation, not just a code dump.

## Contract-Specific Goals

- Compare scheduled hours to actual hours by employee.
- Sum overtime by employee.
- Sum overtime by team.
- Name the team carrying the heaviest overtime load.

## Expected Deliverables

- An employee hours-gap table.
- An overtime ranking by person.
- An overtime ranking by team.
- A final staffing verdict naming the most overloaded team.

## Evaluation Criteria

- Collaboration: role alternation, listening, and requirement handoff stay clear throughout the session.
- Depth: the pair uses advanced T-SQL patterns such as CTEs, subqueries, ranking, pivots, or performance thinking.
- Innovation: the solution goes beyond a first-pass answer and explores edge cases, tuning, or richer analysis.
- Documentation: the notebook captures role notes, query evolution, evidence, and final recommendations.

## Sample Database Schema

### `teams`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |

### `employees`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |
| team_id | INTEGER |  |

### `schedules`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| employee_id | INTEGER |  |
| shift_hours | INTEGER |  |

### `clock_logs`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| employee_id | INTEGER |  |
| hours_worked | INTEGER |  |


In [28]:
-- Session 8: The Overtime Apparition
-- Shared T-SQL workspace
-- Record the team's query evolution, tests, and refinements below.

-- Employee Hours Gap
-- Shows the schedule gap for every employee.
SELECT e.name, t.name AS team_name, s.shift_hours, c.hours_worked,
  c.hours_worked - s.shift_hours AS overtime_hours
FROM employees e
JOIN teams t ON t.id = e.team_id
JOIN schedules s ON s.employee_id = e.id
JOIN clock_logs c ON c.employee_id = e.id
ORDER BY overtime_hours DESC, e.name;

-- Employee Overtime Totals
-- Shows who individually is carrying the most extra hours.
SELECT e.name, SUM(c.hours_worked - s.shift_hours) AS overtime_hours
FROM employees e
JOIN schedules s ON s.employee_id = e.id
JOIN clock_logs c ON c.employee_id = e.id
GROUP BY e.id, e.name
ORDER BY overtime_hours DESC, e.name;

-- Team Overtime Totals
-- Rolls overtime up to the staffing team level.
SELECT t.name, SUM(c.hours_worked - s.shift_hours) AS overtime_hours
FROM teams t
JOIN employees e ON e.team_id = t.id
JOIN schedules s ON s.employee_id = e.id
JOIN clock_logs c ON c.employee_id = e.id
GROUP BY t.id, t.name
ORDER BY overtime_hours DESC, t.name;

-- Add the pair's evolving T-SQL here

## Pair Notes and Recommendation

### Navigator Notes
- What business need or requirement was clarified during the session?
- Which edge cases or decision points mattered most?

### Driver Notes
- How did the query evolve from first attempt to final answer?
- Which advanced T-SQL features helped most here?

### Final Recommendation
- Summarize the insight, action item, or business recommendation.
- Note any performance tuning, validation checks, or optional visualization follow-up.

# Paired Real-World Problem-Solving

## Session 9: The Route Profit Atlas

**Contract ID:** dc009  
**Lead Role This Session:** Navigator  
**Recommended Session Length:** 1-2 hours  
**Notebook Format:** Shared SQL notebook for navigator and driver collaboration

## Business Scenario

Shipping revenue looks healthy, but route margins say otherwise. The pair must identify which lane is quietly destroying profit once transport costs are fully loaded.

## Role Structure

- Navigator acts as the business stakeholder and refines the requirement verbally.
- Driver acts as the programmer or analyst and writes the evolving T-SQL.
- Partners should switch emphasis midway if that improves balance or clarity.

- Navigator note: Define the profit story clearly: revenue minus total route cost, not just revenue alone.
- Driver note: Join revenue to route costs, calculate net margin, and isolate the route that needs to be rerouted or repriced first.

## Session Process

- Navigator states the business need, sharpens the question, and explains what a useful answer would change.
- Driver writes T-SQL in a shared notebook, tests results, and narrates what each iteration proves or disproves.
- Both partners refine the query path together, switch emphasis when needed, and keep role notes in the notebook.
- Finish with recommendations, query evolution, and a notebook trail strong enough for GitHub submission.

## Session Requirements

- Alternate navigator and driver responsibilities across sessions.
- Use advanced T-SQL patterns to move from raw data to a real recommendation.
- Capture role notes, query evolution, results, and a final recommendation in one shared notebook.
- Treat the work like a real business-technical conversation, not just a code dump.

## Contract-Specific Goals

- Aggregate shipment revenue by route.
- Aggregate total cost by route.
- Calculate net margin by route.
- Name the route that needs intervention first.

## Expected Deliverables

- A revenue view by route.
- A cost view by route.
- A net margin ranking.
- A final route intervention recommendation.

## Evaluation Criteria

- Collaboration: role alternation, listening, and requirement handoff stay clear throughout the session.
- Depth: the pair uses advanced T-SQL patterns such as CTEs, subqueries, ranking, pivots, or performance thinking.
- Innovation: the solution goes beyond a first-pass answer and explores edge cases, tuning, or richer analysis.
- Documentation: the notebook captures role notes, query evolution, evidence, and final recommendations.

## Sample Database Schema

### `routes`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |

### `shipments`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| route_id | INTEGER |  |
| revenue | INTEGER |  |

### `route_costs`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| route_id | INTEGER |  |
| fuel_cost | INTEGER |  |
| crew_cost | INTEGER |  |
| toll_cost | INTEGER |  |


In [29]:
-- Session 9: The Route Profit Atlas
-- Shared T-SQL workspace
-- Record the team's query evolution, tests, and refinements below.

-- Revenue by Route
-- Shows the tempting top-line story before cost is loaded in.
SELECT r.name, SUM(s.revenue) AS total_revenue
FROM routes r
JOIN shipments s ON s.route_id = r.id
GROUP BY r.id, r.name
ORDER BY total_revenue DESC;

-- Cost by Route
-- Shows the operating burden for each lane.
SELECT r.name, rc.fuel_cost + rc.crew_cost + rc.toll_cost AS total_cost
FROM routes r
JOIN route_costs rc ON rc.route_id = r.id
ORDER BY total_cost DESC;

-- Net Margin by Route
-- Shows which lane hurts profit once cost is included.
WITH revenue AS (
  SELECT route_id, SUM(revenue) AS total_revenue
  FROM shipments
  GROUP BY route_id
)
SELECT r.name, rev.total_revenue,
  rc.fuel_cost + rc.crew_cost + rc.toll_cost AS total_cost,
  rev.total_revenue - (rc.fuel_cost + rc.crew_cost + rc.toll_cost) AS net_margin
FROM routes r
JOIN revenue rev ON rev.route_id = r.id
JOIN route_costs rc ON rc.route_id = r.id
ORDER BY net_margin ASC, r.name;

-- Add the pair's evolving T-SQL here

## Pair Notes and Recommendation

### Navigator Notes
- What business need or requirement was clarified during the session?
- Which edge cases or decision points mattered most?

### Driver Notes
- How did the query evolve from first attempt to final answer?
- Which advanced T-SQL features helped most here?

### Final Recommendation
- Summarize the insight, action item, or business recommendation.
- Note any performance tuning, validation checks, or optional visualization follow-up.

# Paired Real-World Problem-Solving

## Session 10: The Defect Council

**Contract ID:** dc010  
**Lead Role This Session:** Driver  
**Recommended Session Length:** 1-2 hours  
**Notebook Format:** Shared SQL notebook for navigator and driver collaboration

## Business Scenario

Quality failures are creeping into inventory. The pair must compare suppliers on defects, rework, and claims cost to identify which source is quietly poisoning downstream margin.

## Role Structure

- Navigator acts as the business stakeholder and refines the requirement verbally.
- Driver acts as the programmer or analyst and writes the evolving T-SQL.
- Partners should switch emphasis midway if that improves balance or clarity.

- Navigator note: Define the standard up front: raw defects are not enough; rework and claims cost must influence the verdict.
- Driver note: Join lots, inspections, and claims into a defect-adjusted cost view, then isolate the supplier with the worst downstream impact.

## Session Process

- Navigator states the business need, sharpens the question, and explains what a useful answer would change.
- Driver writes T-SQL in a shared notebook, tests results, and narrates what each iteration proves or disproves.
- Both partners refine the query path together, switch emphasis when needed, and keep role notes in the notebook.
- Finish with recommendations, query evolution, and a notebook trail strong enough for GitHub submission.

## Session Requirements

- Alternate navigator and driver responsibilities across sessions.
- Use advanced T-SQL patterns to move from raw data to a real recommendation.
- Capture role notes, query evolution, results, and a final recommendation in one shared notebook.
- Treat the work like a real business-technical conversation, not just a code dump.

## Contract-Specific Goals

- Aggregate defects by supplier.
- Aggregate rework and claims cost by supplier.
- Calculate a defect-adjusted cost signal by supplier.
- Name the supplier with the worst quality impact.

## Expected Deliverables

- A defect-volume table.
- A rework and claims cost table.
- A defect-adjusted supplier ranking.
- A final supplier quality verdict.

## Evaluation Criteria

- Collaboration: role alternation, listening, and requirement handoff stay clear throughout the session.
- Depth: the pair uses advanced T-SQL patterns such as CTEs, subqueries, ranking, pivots, or performance thinking.
- Innovation: the solution goes beyond a first-pass answer and explores edge cases, tuning, or richer analysis.
- Documentation: the notebook captures role notes, query evolution, evidence, and final recommendations.

## Sample Database Schema

### `suppliers`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| name | TEXT |  |

### `lots`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| supplier_id | INTEGER |  |
| units_received | INTEGER |  |

### `inspections`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| lot_id | INTEGER |  |
| defects_found | INTEGER |  |
| rework_cost | INTEGER |  |

### `claims`
| Column | Type | PK |
|--------|------|----|
| id | INTEGER | Yes |
| lot_id | INTEGER |  |
| claim_amount | INTEGER |  |


In [30]:
-- Session 10: The Defect Council
-- Shared T-SQL workspace
-- Record the team's query evolution, tests, and refinements below.

-- Defects by Supplier
-- Shows raw defect volume by supplier.
SELECT s.name, SUM(i.defects_found) AS total_defects
FROM suppliers s
JOIN lots l ON l.supplier_id = s.id
JOIN inspections i ON i.lot_id = l.id
GROUP BY s.id, s.name
ORDER BY total_defects DESC;

-- Quality Cost by Supplier
-- Adds the financial cost of poor quality to the picture.
SELECT s.name,
  SUM(i.rework_cost) AS total_rework_cost,
  SUM(c.claim_amount) AS total_claim_cost
FROM suppliers s
JOIN lots l ON l.supplier_id = s.id
JOIN inspections i ON i.lot_id = l.id
JOIN claims c ON c.lot_id = l.id
GROUP BY s.id, s.name
ORDER BY total_rework_cost + total_claim_cost DESC;

-- Defect-Adjusted Impact
-- Combines the operational and financial side of supplier quality.
SELECT s.name,
  SUM(i.defects_found) AS total_defects,
  SUM(i.rework_cost + c.claim_amount) AS total_quality_cost
FROM suppliers s
JOIN lots l ON l.supplier_id = s.id
JOIN inspections i ON i.lot_id = l.id
JOIN claims c ON c.lot_id = l.id
GROUP BY s.id, s.name
ORDER BY total_quality_cost DESC, total_defects DESC;

-- Add the pair's evolving T-SQL here

## Pair Notes and Recommendation

### Navigator Notes
- What business need or requirement was clarified during the session?
- Which edge cases or decision points mattered most?

### Driver Notes
- How did the query evolve from first attempt to final answer?
- Which advanced T-SQL features helped most here?

### Final Recommendation
- Summarize the insight, action item, or business recommendation.
- Note any performance tuning, validation checks, or optional visualization follow-up.